In [ ]:

import pandas as pd
import zipfile
import os
from google.colab import files

# Upload your ZIP file
uploaded = files.upload()

# Extract uploaded ZIP files
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall("cricket_data")

print("Dataset uploaded and extracted!")


KeyboardInterrupt: 

In [ ]:

import pandas as pd

# Inspect all uploaded CSV files
for filename in uploaded.keys():

    df = pd.read_csv(filename, nrows=5)

    print("\n" + "="*60)
    print("FILE:", filename)
    print("COLUMNS:", list(df.columns))
    print("\nSAMPLE RECORDS:")
    display(df.head())


In [ ]:

import pandas as pd
import os

for filename in uploaded.keys():

    print("\n" + "="*65)
    print("FILE:", filename)

    total_rows = 0
    missing_values = None

    for chunk in pd.read_csv(filename, chunksize=100000):

        total_rows += len(chunk)

        if missing_values is None:
            missing_values = chunk.isnull().sum()
        else:
            missing_values += chunk.isnull().sum()

    print("TOTAL ROWS:", total_rows)
    print("TOTAL COLUMNS:", len(missing_values))
    print("\nMISSING VALUES:")
    print(missing_values)


In [ ]:

import pandas as pd

df = pd.read_csv("ball_by_ball.csv")

# 1. Check duplicate records
print("EXACT DUPLICATE ROWS:", df.duplicated().sum())

# 2. Check duplicate delivery identifiers
key = ["match_id", "innings", "innings_ball_num"]

print("\nDUPLICATE DELIVERY IDENTIFIERS:",
      df.duplicated(subset=key).sum())

# 3. Check data types
print("\nDATA TYPES:")
print(df.dtypes)

# 4. Check number of unique matches and players
print("\nUNIQUE MATCHES:", df["match_id"].nunique())
print("UNIQUE BATTERS:", df["striker_id"].nunique())
print("UNIQUE BOWLERS:", df["bowler_id"].nunique())

# 5. Check innings distribution
print("\nINNINGS DISTRIBUTION:")
print(df["innings"].value_counts().sort_index())

# 6. Check delivery extras
print("\nEXTRAS DISTRIBUTION:")
print(df["extras_type"].value_counts(dropna=False))



In [ ]:

import pandas as pd

# Load both datasets
balls = pd.read_csv("ball_by_ball.csv")
matches = pd.read_csv("matches.csv")

# 1. Find matches missing from ball-by-ball data
ball_matches = set(balls["match_id"])
official_matches = set(matches["match_id"])

missing_matches = official_matches - ball_matches

print("MATCHES WITHOUT BALL-BY-BALL DATA:", len(missing_matches))
print("MISSING MATCH IDs:", sorted(missing_matches))

# 2. Investigate innings 3 and 4
extra_innings = balls[balls["innings"] > 2]

print("\nEXTRA INNINGS RECORDS:", len(extra_innings))
print("\nMATCHES WITH INNINGS 3 AND 4:")
print(extra_innings.groupby(["match_id", "innings"]).size())

# 3. Check unusual innings numbers
print("\nALL INNINGS DISTRIBUTION:")
print(balls["innings"].value_counts().sort_index())

# 4. Check extras and their associated runs
print("\nEXTRAS BY TYPE:")
print(
    balls.groupby("extras_type", dropna=False)
    [["runs", "extras1_runs", "extras_2_runs"]]
    .agg(["count", "sum"])
)

# 5. Check match IDs not found in matches.csv
orphan_matches = ball_matches - official_matches

print("\nBALL-BY-BALL MATCHES NOT IN MATCHES FILE:",
      len(orphan_matches))
print(sorted(orphan_matches))


In [ ]:

import pandas as pd

balls = pd.read_csv("ball_by_ball.csv")
matches = pd.read_csv("matches.csv")

# A. Investigate the 16 missing matches
missing_ids = [
    31281, 31284, 31288, 32610, 32612, 32622,
    36210, 36564, 38797, 40761, 41502, 41503,
    42030, 46008, 48513, 50189
]

print("========== MISSING MATCH DETAILS ==========")

display(
    matches[matches["match_id"].isin(missing_ids)]
    [["match_id", "comp_id", "match_date", "stage",
      "home_team", "away_team", "result", "win_team_id"]]
)

# B. Investigate matches with additional innings
print("\n========== EXTRA INNINGS MATCH DETAILS ==========")

extra_ids = [34429, 40725, 45512, 48027, 48029, 50359]

display(
    matches[matches["match_id"].isin(extra_ids)]
)

# C. Inspect actual additional innings deliveries
print("\n========== ADDITIONAL INNINGS SAMPLE ==========")

display(
    balls[balls["innings"] > 2]
    [["match_id", "innings", "overid", "over_ball_num",
      "striker_name", "bowler_name", "runs",
      "is_wicket", "extras_type"]]
    .sort_values(["match_id", "innings", "innings_ball_num"])
)

# D. Understand extras columns
print("\n========== EXTRAS SAMPLE ==========")

display(
    balls[balls["extras_type"].notna()]
    [["extras_type", "runs", "extras1_runs", "extras_2_runs"]]
    .head(25)
)


In [ ]:

import pandas as pd

# Load a fresh copy
balls_diag = pd.read_csv("ball_by_ball.csv")

print("COLUMNS IN DATASET:")
print(balls_diag.columns.tolist())

print("\nADDITIONAL INNINGS SAMPLE:")

extra_innings = balls_diag[balls_diag["innings"] > 2]

cols = [
    "match_id", "innings", "overid", "over_ball_num",
    "striker_name", "bowler_name", "runs",
    "is_wicket", "extras_type"
]

print(
    extra_innings[cols]
    .sort_values(["match_id", "innings", "overid", "over_ball_num"])
    .to_string(index=False)
)

print("\nEXTRAS SAMPLE:")

print(
    balls_diag.loc[
        balls_diag["extras_type"].notna(),
        ["extras_type", "runs", "extras1_runs", "extras_2_runs"]
    ].head(30).to_string(index=False)
)


In [ ]:

# Examine all scoring combinations
print("SCORING COLUMN SUMMARY")

print(
    balls_diag.groupby("extras_type", dropna=False)
    [["runs", "extras1_runs", "extras_2_runs"]]
    .agg(["count", "sum", "min", "max"])
    .to_string()
)

print("\nNORMAL DELIVERY EXAMPLES")

print(
    balls_diag.loc[
        balls_diag["extras_type"].isna(),
        ["runs", "is_four", "is_six", "is_wicket",
         "extras1_runs", "extras_2_runs"]
    ].head(20).to_string(index=False)
)

print("\nEXTRAS TYPE COUNTS")

print(balls_diag["extras_type"].value_counts(dropna=False))


In [ ]:

print("EXTRAS ENCODING CHECK")

print(
    pd.crosstab(
        balls_diag["extras_type"].fillna("normal"),
        balls_diag["extras1_runs"],
        dropna=False
    )
)

print("\nNON-ZERO extras_2_runs RECORDS")

print(
    balls_diag.loc[
        balls_diag["extras_2_runs"] != 0,
        ["extras_type", "runs", "extras1_runs", "extras_2_runs"]
    ].head(40).to_string(index=False)
)

print("\nEXTRAS WITH NON-ZERO runs")

print(
    balls_diag.loc[
        balls_diag["extras_type"].notna() & (balls_diag["runs"] != 0),
        ["extras_type", "runs", "extras1_runs", "extras_2_runs"]
    ].to_string(index=False)
)


In [ ]:

print("RUN DISTRIBUTION BY DELIVERY TYPE")

print(
    pd.crosstab(
        balls_diag["extras_type"].fillna("normal"),
        balls_diag["runs"],
        margins=True
    )
)

print("\nBOUNDARY FLAG VALIDATION")

print(
    balls_diag.groupby("extras_type", dropna=False)
    [["is_four", "is_six", "is_wicket"]]
    .sum()
)

print("\nRECORDS WITH extras_2_runs = 1 OR 4")

print(
    balls_diag.loc[
        balls_diag["extras_2_runs"].isin([1, 4]),
        ["match_id", "innings", "overid", "over_ball_num",
         "striker_name", "runs", "extras_type",
         "extras1_runs", "extras_2_runs"]
    ].head(25).to_string(index=False)
)


In [ ]:

# Inspect all non-zero extras_2_runs values
print("EXTRAS_2_RUNS VALUE DISTRIBUTION")

print(
    balls_diag.groupby(
        ["extras_type", "extras_2_runs"],
        dropna=False
    ).size().to_string()
)

# Check whether extras_2_runs is associated with batter runs
print("\nEXTRAS_2_RUNS WITH BATTER RUNS")

print(
    balls_diag.loc[
        balls_diag["extras_2_runs"] != 0,
        ["extras_type", "runs", "extras1_runs",
         "extras_2_runs", "is_four", "is_six"]
    ].groupby(
        ["extras_type", "extras_2_runs", "runs"],
        dropna=False
    ).size().to_string()
)

# Inspect matches.csv scoring-related columns
matches_diag = pd.read_csv("matches.csv")

print("\nMATCHES CSV COLUMNS")
print(matches_diag.columns.tolist())


In [ ]:
print("FOUR-RUN DELIVERIES")
print(
    balls_diag[
        (balls_diag["runs"] == 4) |
        (balls_diag["is_four"] == True)
    ].groupby(["extras_type", "runs", "is_four"], dropna=False)
    .size().to_string()
)

print("\nSIX-RUN DELIVERIES")
print(
    balls_diag[
        (balls_diag["runs"] == 6) |
        (balls_diag["is_six"] == True)
    ].groupby(["extras_type", "runs", "is_six"], dropna=False)
    .size().to_string()
)

In [ ]:
matches_diag = pd.read_csv("matches.csv")
balls_diag = pd.read_csv("ball_by_ball.csv")

print("COMPETITION IDs")
print(matches_diag["comp_id"].value_counts().to_string())

print("\nMATCH RESULTS")
print(matches_diag["result"].value_counts().to_string())

print("\nMATCH STAGES")
print(matches_diag["stage"].value_counts().to_string())

print("\nMATCH DATE RANGE")
print(matches_diag["match_date"].min())
print(matches_diag["match_date"].max())

In [ ]:
competitions_diag = pd.read_csv("competitions.csv")

print("COMPETITIONS FILE COLUMNS:")
print(competitions_diag.columns.tolist())

print("\nCOMPETITION ID MAPPING:")
print(
    matches_diag[["comp_id"]].drop_duplicates()
    .merge(competitions_diag, on="comp_id", how="left")
    .sort_values("comp_id")
    .to_string(index=False)
)

In [ ]:

import pandas as pd

competitions = pd.read_csv("competitions.csv")
matches = pd.read_csv("matches.csv")

# Count matches available in each competition
competition_summary = (
    matches.groupby("comp_id")
    .agg(
        matches_available=("match_id", "nunique"),
        first_match=("match_date", "min"),
        last_match=("match_date", "max")
    )
    .reset_index()
)

# Attach competition names and formats
competition_summary = competition_summary.merge(
    competitions,
    on="comp_id",
    how="left"
)

# Show all T20 and T20I competitions
t20_competitions = competition_summary[
    competition_summary["match_format"].isin(["T20", "T20I"])
].sort_values(["match_format", "season", "comp_name"])

print(t20_competitions.to_string(index=False))


In [ ]:

import pandas as pd

matches = pd.read_csv("matches.csv")
competitions = pd.read_csv("competitions.csv")
balls = pd.read_csv("ball_by_ball.csv")

# IPL competitions
ipl_ids = [1358, 1715, 2193, 2725, 3298]

# Emerging competitions
emerging_ids = [2670, 3289]

# Performance competitions only
performance_ids = ipl_ids + emerging_ids

# T20I competitions kept separately for eligibility checks
t20i_ids = competitions.loc[
    competitions["match_format"] == "T20I",
    "comp_id"
].tolist()

# Match metadata
performance_matches = matches[
    matches["comp_id"].isin(performance_ids)
].copy()

t20i_matches = matches[
    matches["comp_id"].isin(t20i_ids)
].copy()

# Ball-by-ball performance data
performance_match_ids = performance_matches["match_id"].unique()

performance_balls = balls[
    (balls["match_id"].isin(performance_match_ids)) &
    (balls["innings"].isin([1, 2]))
].copy()

print("PERFORMANCE MATCHES:", len(performance_matches))
print("PERFORMANCE DELIVERIES:", len(performance_balls))
print("T20I MATCHES FOR ELIGIBILITY CHECK:", len(t20i_matches))

print("\nCOMPETITION-WISE MATCHES:")
print(
    performance_matches.groupby("comp_id")["match_id"]
    .nunique()
)

print("\nPLAYER TABLE STRUCTURE:")
playing = pd.read_csv("playing_eleven.csv")
mapping = pd.read_csv("competition_team_player_mapping.csv")

print("Playing eleven columns:", playing.columns.tolist())
print("Player mapping columns:", mapping.columns.tolist())
print("\nPlaying eleven sample:")
print(playing.head(3).to_string(index=False))
print("\nPlayer mapping sample:")
print(mapping.head(3).to_string(index=False))


In [ ]:

# Create a new candidate list without the 10-match IPL restriction

candidates_all = player_pool[
    player_pool["t20i_matches_recorded"] == 0
].copy()

candidates_all = candidates_all.sort_values(
    ["emerging_matches", "ipl_matches_recorded"],
    ascending=[False, True]
)

print("TOTAL PLAYERS:", len(player_pool))
print("NEW PROVISIONAL CANDIDATES:", len(candidates_all))

print("\nCANDIDATE LIST:")
print(candidates_all.to_string(index=False))

# Save separately; previous candidate file remains untouched
candidates_all.to_csv(
    "provisional_candidates_without_ipl_limit.csv",
    index=False
)


In [ ]:

import pandas as pd

# Reload required files
matches = pd.read_csv("matches.csv")
competitions = pd.read_csv("competitions.csv")
playing = pd.read_csv("playing_eleven.csv")
mapping = pd.read_csv("competition_team_player_mapping.csv")

# Competition IDs
ipl_ids = [1358, 1715, 2193, 2725, 3298]
emerging_ids = [2670, 3289]

t20i_ids = competitions.loc[
    competitions["match_format"] == "T20I", "comp_id"
].tolist()

# Match IDs
ipl_match_ids = matches.loc[
    matches["comp_id"].isin(ipl_ids), "match_id"
].unique()

emerging_match_ids = matches.loc[
    matches["comp_id"].isin(emerging_ids), "match_id"
].unique()

t20i_match_ids = matches.loc[
    matches["comp_id"].isin(t20i_ids), "match_id"
].unique()

# Only actual playing appearances
played = playing[
    playing["is_played"].astype(str).str.lower().eq("true")
].copy()

# Count unique appearances
ipl_apps = (
    played[played["match_id"].isin(ipl_match_ids)]
    .groupby("player_id")["match_id"].nunique()
    .rename("ipl_matches_recorded")
)

emerging_apps = (
    played[played["match_id"].isin(emerging_match_ids)]
    .groupby("player_id")["match_id"].nunique()
    .rename("emerging_matches")
)

t20i_apps = (
    played[played["match_id"].isin(t20i_match_ids)]
    .groupby("player_id")["match_id"].nunique()
    .rename("t20i_matches_recorded")
)

# Player names
player_names = (
    mapping[mapping["comp_id"].isin(ipl_ids + emerging_ids)]
    [["player_id", "player_name"]]
    .drop_duplicates("player_id")
)

# Build player pool
player_pool = (
    player_names
    .merge(ipl_apps, on="player_id", how="left")
    .merge(emerging_apps, on="player_id", how="left")
    .merge(t20i_apps, on="player_id", how="left")
)

player_pool = player_pool.fillna({
    "ipl_matches_recorded": 0,
    "emerging_matches": 0,
    "t20i_matches_recorded": 0
})

# No IPL appearance restriction
candidates_all = player_pool[
    player_pool["t20i_matches_recorded"] == 0
].copy()

candidates_all = candidates_all.sort_values(
    ["emerging_matches", "ipl_matches_recorded"],
    ascending=[False, True]
)

print("TOTAL PLAYERS:", len(player_pool))
print("PROVISIONAL CANDIDATES:", len(candidates_all))

print("\nCANDIDATE LIST:")
print(candidates_all.to_string(index=False))

candidates_all.to_csv(
    "provisional_candidates_without_ipl_limit.csv",
    index=False
)


In [ ]:
eligible_shortlist = candidates_all[
    ["player_id", "player_name",
     "ipl_matches_recorded", "emerging_matches"]
].copy()

eligible_shortlist = eligible_shortlist.sort_values(
    ["emerging_matches", "ipl_matches_recorded"],
    ascending=False
)

print("PROVISIONAL PLAYER NAMES:", len(eligible_shortlist))

display(eligible_shortlist)

eligible_shortlist.to_csv(
    "provisional_eligible_player_names.csv",
    index=False
)

In [ ]:
active_pool = eligible_shortlist[
    (eligible_shortlist["ipl_matches_recorded"] > 0) |
    (eligible_shortlist["emerging_matches"] > 0)
].copy()

active_pool = active_pool.sort_values(
    ["emerging_matches", "ipl_matches_recorded"],
    ascending=False
)

print("PLAYERS WITH RECORDED MATCH APPEARANCES:", len(active_pool))

display(active_pool)

active_pool.to_csv(
    "players_with_recorded_performances.csv",
    index=False
)

In [ ]:
active_pool[[
    "player_id",
    "player_name",
    "ipl_matches_recorded",
    "emerging_matches"
]].to_csv(
    "248_player_verification_list.csv",
    index=False
)

from IPython.display import display, FileLink
display(FileLink("248_player_verification_list.csv"))

In [ ]:
from google.colab import files

files.download("248_player_verification_list.csv")

In [ ]:

print("Runtime Check")

try:
    print("Ball-by-ball dataset:", ball_by_ball.shape)
except NameError:
    print("Ball-by-ball dataset not loaded")

try:
    print("Matches dataset:", matches.shape)
except NameError:
    print("Matches dataset not loaded")


In [ ]:

import pandas as pd

print("ALL AVAILABLE DATAFRAMES:\n")

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")


In [ ]:

print("ACTIVE PLAYER POOL")
display(active_pool.head())

print("\nPERFORMANCE BALLS COLUMNS")
print(performance_balls.columns.tolist())

print("\nPLAYING DATA COLUMNS")
print(playing.columns.tolist())

print("\nPLAYER MAPPING COLUMNS")
print(mapping.columns.tolist())


In [ ]:

# STEP 1: Batting involvement

batting_stats = (
    performance_balls
    .groupby(['striker_id', 'striker_name'])
    .agg(
        batting_matches=('match_id', 'nunique'),
        batting_deliveries=('match_id', 'size'),
        batting_runs=('runs', 'sum')
    )
    .reset_index()
    .rename(columns={
        'striker_id': 'player_id',
        'striker_name': 'player_name'
    })
)

# STEP 2: Bowling involvement

bowling_stats = (
    performance_balls
    .groupby(['bowler_id', 'bowler_name'])
    .agg(
        bowling_matches=('match_id', 'nunique'),
        bowling_deliveries=('match_id', 'size')
    )
    .reset_index()
    .rename(columns={
        'bowler_id': 'player_id',
        'bowler_name': 'player_name'
    })
)

# STEP 3: Combine batting and bowling statistics

player_contributions = (
    active_pool[['player_id', 'player_name']]
    .merge(
        batting_stats.drop(columns='player_name'),
        on='player_id',
        how='left'
    )
    .merge(
        bowling_stats.drop(columns='player_name'),
        on='player_id',
        how='left'
    )
)

# STEP 4: Replace missing involvement counts with zero

count_columns = [
    'batting_matches',
    'batting_deliveries',
    'batting_runs',
    'bowling_matches',
    'bowling_deliveries'
]

player_contributions[count_columns] = (
    player_contributions[count_columns].fillna(0)
)

# STEP 5: Display results

print("PLAYER CONTRIBUTION SUMMARY")
print("Total players:", len(player_contributions))

display(
    player_contributions
    .sort_values('batting_deliveries', ascending=False)
    .head(20)
)


In [ ]:

# Check IPL team association of all 248 players

ipl_mapping = mapping[
    mapping['comp_name'].astype(str).str.contains(
        'Indian Premier League|IPL',
        case=False,
        na=False,
        regex=True
    )
].copy()

ipl_player_teams = (
    ipl_mapping[
        ipl_mapping['player_id'].isin(active_pool['player_id'])
    ]
    .groupby(['player_id', 'player_name'])
    .agg(
        ipl_teams=('team_name', lambda x: ', '.join(sorted(set(x.dropna()))))
    )
    .reset_index()
)

print("Players with IPL team records:", len(ipl_player_teams))

display(ipl_player_teams.head(20))


In [ ]:

# Create uncapped verification worksheet

verification_sheet = (
    active_pool[['player_id', 'player_name']]
    .merge(ipl_player_teams, on=['player_id', 'player_name'], how='left')
    .merge(player_contributions, on=['player_id', 'player_name'], how='left',
           suffixes=('', '_stats'))
)

verification_sheet['ipl_teams'] = verification_sheet['ipl_teams'].fillna('No IPL team record')

# Manual verification columns
verification_sheet['Test_appearances'] = ''
verification_sheet['ODI_appearances'] = ''
verification_sheet['T20I_appearances_all_time'] = ''
verification_sheet['Uncapped_verified'] = ''
verification_sheet['Verification_source'] = ''

verification_sheet = verification_sheet.sort_values(
    'player_name'
).reset_index(drop=True)

verification_sheet.to_csv(
    'uncapped_verification_sheet.csv',
    index=False
)

print("Total players:", len(verification_sheet))
display(verification_sheet.head(15))


In [ ]:

# Check available international competition data

print("Available competition columns:")
print(competitions.columns.tolist())

print("\nCompetition names containing international formats:")
international_competitions = competitions[
    competitions.astype(str).apply(
        lambda col: col.str.contains(
            r'Test|ODI|T20I|International',
            case=False,
            na=False
        )
    ).any(axis=1)
]

display(international_competitions.head(40))
print("Matching competitions:", len(international_competitions))


In [ ]:

print("Matches columns:")
print(matches.columns.tolist())

print("\nPossible date columns:")
print([
    col for col in matches.columns
    if 'date' in col.lower() or 'time' in col.lower()
])


In [ ]:

# Check international appearances during 2022–2026

international_ids = competitions[
    competitions['match_format'].astype(str).str.upper().isin(
        ['TEST', 'ODI', 'T20I']
    )
][['comp_id', 'match_format']]

international_matches = matches.merge(
    international_ids,
    on='comp_id',
    how='inner'
)

international_players = playing.merge(
    international_matches[
        ['match_id', 'comp_id', 'match_format', 'match_date']
    ],
    on='match_id',
    how='inner'
)

pool_international = international_players[
    international_players['player_id'].isin(active_pool['player_id'])
].copy()

international_summary = (
    pool_international.groupby('player_id')
    .agg(
        international_matches=('match_id', 'nunique'),
        formats=('match_format', lambda x: ', '.join(sorted(set(x)))),
        last_international_match=('match_date', 'max')
    )
    .reset_index()
)

international_check = (
    active_pool[['player_id', 'player_name']]
    .merge(international_summary, on='player_id', how='left')
)

international_check['international_matches'] = (
    international_check['international_matches'].fillna(0).astype(int)
)

international_check['formats'] = international_check['formats'].fillna(
    'No recorded appearance'
)

international_check = international_check.sort_values(
    'last_international_match',
    ascending=False,
    na_position='last'
)

display(international_check.head(30))

print("Players with recorded international appearances:",
      (international_check['international_matches'] > 0).sum())

print("Players with no recorded appearance:",
      (international_check['international_matches'] == 0).sum())


In [ ]:

# Inspect all international-format competition names

international_competitions = competitions[
    competitions['match_format'].astype(str).str.upper().isin(
        ['TEST', 'ODI', 'T20I']
    )
].copy()

display(
    international_competitions[
        ['comp_id', 'comp_name', 'match_format', 'season']
    ].sort_values(['match_format', 'season', 'comp_name'])
)


In [ ]:

# Identify emerging and development competitions

emerging_competitions = competitions[
    competitions['comp_name'].astype(str).str.contains(
        r'Emerging|Rising Stars|Development',
        case=False,
        na=False
    )
][['comp_id', 'comp_name', 'match_format', 'season']]

display(emerging_competitions)

print("Emerging/development competitions found:",
      len(emerging_competitions))


In [ ]:

# Exclude Emerging and Rising Stars competitions from international eligibility

excluded_comp_ids = [2670, 3289]

senior_international_ids = competitions[
    competitions['match_format'].astype(str).str.upper().isin(
        ['TEST', 'ODI', 'T20I']
    )
    & ~competitions['comp_id'].isin(excluded_comp_ids)
][['comp_id', 'match_format']]

senior_international_matches = matches.merge(
    senior_international_ids,
    on='comp_id',
    how='inner'
)

senior_international_players = playing.merge(
    senior_international_matches[
        ['match_id', 'match_format', 'match_date']
    ],
    on='match_id',
    how='inner'
)

pool_senior_international = senior_international_players[
    senior_international_players['player_id'].isin(
        active_pool['player_id']
    )
].copy()

senior_summary = (
    pool_senior_international.groupby('player_id')
    .agg(
        senior_international_matches=('match_id', 'nunique'),
        formats=('match_format', lambda x: ', '.join(sorted(set(x)))),
        last_senior_international=('match_date', 'max')
    )
    .reset_index()
)

senior_check = (
    active_pool[['player_id', 'player_name']]
    .merge(senior_summary, on='player_id', how='left')
)

senior_check['senior_international_matches'] = (
    senior_check['senior_international_matches'].fillna(0).astype(int)
)

senior_check['formats'] = senior_check['formats'].fillna(
    'No recorded appearance'
)

senior_check = senior_check.sort_values(
    'last_senior_international',
    ascending=False,
    na_position='last'
)

display(senior_check.head(30))

print("Players with senior international appearances:",
      (senior_check['senior_international_matches'] > 0).sum())

print("Players with no recorded senior international appearances:",
      (senior_check['senior_international_matches'] == 0).sum())


In [ ]:

# Final 5-year eligibility classification for IPL 2027

cutoff_date = pd.Timestamp('2021-11-10')

senior_check['last_senior_international'] = pd.to_datetime(
    senior_check['last_senior_international'],
    errors='coerce'
)

senior_check['eligibility_status'] = 'Eligible - no recorded senior international appearance'

senior_check.loc[
    senior_check['last_senior_international'] > cutoff_date,
    'eligibility_status'
] = 'Not eligible - international appearance after cutoff'

senior_check.loc[
    senior_check['last_senior_international'].notna()
    & (senior_check['last_senior_international'] <= cutoff_date),
    'eligibility_status'
] = 'Eligible - last international appearance before cutoff'

display(
    senior_check['eligibility_status']
    .value_counts()
    .rename_axis('eligibility_status')
    .reset_index(name='players')
)


In [ ]:

# Create provisional eligible player pool

eligible_pool = senior_check[
    senior_check['eligibility_status'].str.startswith('Eligible -')
].copy()

# Add batting and bowling contribution data
eligible_contributions = eligible_pool[
    ['player_id', 'player_name', 'eligibility_status']
].merge(
    player_contributions,
    on=['player_id', 'player_name'],
    how='left'
)

# Identify players with recorded batting involvement
batting_candidates = eligible_contributions[
    eligible_contributions['batting_deliveries'] > 0
].copy()

# Sort by batting deliveries for initial inspection
batting_candidates = batting_candidates.sort_values(
    ['batting_deliveries', 'batting_runs'],
    ascending=False
)

print("Eligible players:", len(eligible_pool))
print("Players with batting involvement:", len(batting_candidates))
print("Players with no recorded batting deliveries:",
      len(eligible_pool) - len(batting_candidates))

display(
    batting_candidates[
        ['player_name', 'batting_matches', 'batting_deliveries',
         'batting_runs', 'bowling_matches', 'bowling_deliveries']
    ].head(30)
)


In [ ]:
# Count IPL appearances for every player in our eligible pool

ipl_match_ids = performance_matches[
    performance_matches['comp_id'].isin([1358, 1715, 2193, 2725, 3298])
]['match_id'].unique()

ipl_playing = playing[
    (playing['match_id'].isin(ipl_match_ids)) &
    (playing['is_played'] == 1)
].copy()

ipl_appearances = (
    ipl_playing.groupby('player_id')['match_id']
    .nunique()
    .reset_index(name='ipl_matches_in_dataset')
)

ipl_appearances = (
    eligible_pool[['player_id', 'player_name']]
    .merge(ipl_appearances, on='player_id', how='left')
)

ipl_appearances['ipl_matches_in_dataset'] = (
    ipl_appearances['ipl_matches_in_dataset'].fillna(0).astype(int)
)

display(
    ipl_appearances.sort_values(
        'ipl_matches_in_dataset', ascending=False
    ).head(30)
)

print("Players with more than 10 IPL matches in dataset:",
      (ipl_appearances['ipl_matches_in_dataset'] > 10).sum())

In [ ]:
# Prepare player list for career IPL match verification

career_check = eligible_pool[
    ['player_id', 'player_name']
].drop_duplicates().copy()

career_check = career_check.merge(
    ipl_appearances[
        ['player_id', 'ipl_matches_in_dataset']
    ],
    on='player_id',
    how='left'
)

career_check['career_ipl_matches'] = pd.NA
career_check['verification_status'] = 'Pending external verification'

career_check.to_csv(
    'career_ipl_matches_verification.csv',
    index=False
)

print("Players ready for verification:", len(career_check))
display(career_check.head(15))

In [ ]:
# Create a working verification sheet

career_check = eligible_pool[
    ['player_id', 'player_name']
].drop_duplicates().copy()

career_check = career_check.merge(
    ipl_appearances[
        ['player_id', 'ipl_matches_in_dataset']
    ],
    on='player_id',
    how='left'
)

career_check['career_ipl_matches'] = pd.NA

career_check.to_csv(
    'career_ipl_matches_verification.csv',
    index=False
)

print("Total players:", len(career_check))
display(career_check.head(10))

In [ ]:
import requests
import zipfile
import io
import json
import pandas as pd
import re
import unicodedata
from collections import Counter

# Download complete IPL historical JSON data
url = "https://cricsheet.org/downloads/ipl_json.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()

ipl_zip = zipfile.ZipFile(io.BytesIO(response.content))

print("Downloaded files:", len(ipl_zip.namelist()))
print("Download size:", round(len(response.content) / (1024 * 1024), 2), "MB")

In [ ]:
# Extract player appearances from historical IPL records

from collections import Counter
import json
import re
import unicodedata

career_appearances = Counter()

for file_name in ipl_zip.namelist():
    if not file_name.endswith('.json'):
        continue

    with ipl_zip.open(file_name) as f:
        match_data = json.load(f)

    info = match_data.get('info', {})
    registry = info.get('registry', {}).get('people', {})

    match_players = set()

    for team, players in info.get('players', {}).items():
        match_players.update(players)

    for player_name in match_players:
        career_appearances[player_name] += 1

print("Historical IPL matches processed:", len(ipl_zip.namelist()))
print("Unique player names found:", len(career_appearances))

display(
    pd.DataFrame(
        career_appearances.items(),
        columns=['player_name', 'career_ipl_matches']
    ).sort_values('career_ipl_matches', ascending=False).head(15)
)

In [ ]:
# Inspect one historical IPL match

sample_file = next(
    name for name in ipl_zip.namelist()
    if name.endswith('.json')
)

with ipl_zip.open(sample_file) as f:
    sample_match = json.load(f)

print("Sample file:", sample_file)
print("Match date:", sample_match.get('meta', {}).get('data_version'))
print("\nTeams and registered players:")

for team, players in sample_match.get('info', {}).get('players', {}).items():
    print(team, ":", len(players), "players")
    print(players)

print("\nInnings available:", len(sample_match.get('innings', [])))

In [ ]:
from difflib import get_close_matches
import re
import unicodedata

def normalize_name(name):
    name = unicodedata.normalize('NFKD', str(name))
    name = name.encode('ascii', 'ignore').decode('ascii')
    name = re.sub(r'[^a-zA-Z ]', '', name)
    return ' '.join(name.lower().split())

# Historical IPL names
historical_names = list(career_appearances.keys())

# Our 228 eligible players
our_names = career_check[
    ['player_id', 'player_name']
].drop_duplicates().copy()

our_names['normalized_name'] = our_names['player_name'].apply(normalize_name)

historical_lookup = {
    normalize_name(name): name
    for name in historical_names
}

# Exact name matches only
our_names['historical_name'] = our_names['normalized_name'].map(
    historical_lookup
)

exact_matches = our_names[
    our_names['historical_name'].notna()
].copy()

unmatched = our_names[
    our_names['historical_name'].isna()
].copy()

print("Exact matches:", len(exact_matches))
print("Names needing manual verification:", len(unmatched))

# Show possible matches without automatically accepting them
for _, row in unmatched.head(30).iterrows():
    suggestions = get_close_matches(
        row['player_name'],
        historical_names,
        n=3,
        cutoff=0.45
    )
    print(row['player_name'], "->", suggestions)

In [ ]:

# STEP 1: Inspect the cleaned datasets

print("=" * 60)
print("MATCH DATA")
print("=" * 60)

print("Shape:", matches.shape)
print("Columns:", matches.columns.tolist())
display(matches.head(5))


print("\n" + "=" * 60)
print("PLAYER MAPPING DATA")
print("=" * 60)

print("Shape:", mapping.shape)
print("Columns:", mapping.columns.tolist())
display(mapping.head(10))


print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(matches.dtypes)


In [ ]:

import pandas as pd

# ============================================
# IPL 2027 — FINAL ELIGIBLE PLAYER LIST
# ============================================

CUTOFF = pd.Timestamp("2021-11-10")
EXCLUDED_COMP_IDS = [2670, 3289]

# 1. Identify senior international competitions
senior_comps = competitions[
    competitions["match_format"].astype(str).str.upper().isin(
        ["TEST", "ODI", "T20I"]
    )
    & ~competitions["comp_id"].isin(EXCLUDED_COMP_IDS)
].copy()

# 2. Connect competitions with matches
senior_matches = matches[
    ["match_id", "comp_id", "match_date"]
].merge(
    senior_comps[["comp_id", "match_format"]],
    on="comp_id",
    how="inner"
)

senior_matches["match_date"] = pd.to_datetime(
    senior_matches["match_date"],
    errors="coerce"
)

# Only appearances AFTER the cutoff
senior_matches = senior_matches[
    senior_matches["match_date"] > CUTOFF
]

# 3. Identify players who appeared in senior internationals
senior_appearances = playing[
    ["match_id", "player_id"]
].merge(
    senior_matches,
    on="match_id",
    how="inner"
)

senior_check = (
    senior_appearances
    .groupby("player_id")
    .agg(
        senior_international_matches=("match_id", "nunique"),
        formats=("match_format",
                 lambda x: ", ".join(sorted(x.unique()))),
        last_senior_international=("match_date", "max")
    )
    .reset_index()
)

# 4. Build final player pool
final_pool = active_pool.merge(
    senior_check,
    on="player_id",
    how="left"
)

final_pool["senior_international_matches"] = (
    final_pool["senior_international_matches"]
    .fillna(0)
    .astype(int)
)

# 5. Apply eligibility rule
eligible_players = final_pool[
    final_pool["senior_international_matches"] == 0
].copy()

ineligible_players = final_pool[
    final_pool["senior_international_matches"] > 0
].copy()

# 6. Final sorting
eligible_players = eligible_players.sort_values(
    "player_name"
).reset_index(drop=True)

# 7. Export files
eligible_players.to_csv(
    "IPL_2027_Final_Eligible_Players.csv",
    index=False
)

ineligible_players.to_csv(
    "IPL_2027_Ineligible_Players.csv",
    index=False
)

# 8. Final report
print("=" * 60)
print("IPL 2027 — FINAL ELIGIBILITY REPORT")
print("=" * 60)

print("Total Candidate Players:", len(final_pool))
print("Eligible Players:", len(eligible_players))
print("Ineligible Players:", len(ineligible_players))

print("\nFINAL ELIGIBLE PLAYER LIST")
display(eligible_players)

print("\nINELIGIBLE PLAYER LIST")
display(ineligible_players)

print("\nFiles exported successfully!")


In [ ]:
import os

print(os.path.abspath("IPL_2027_Final_Eligible_Players.csv"))
print(os.path.abspath("IPL_2027_Ineligible_Players.csv"))

In [ ]:

from google.colab import files
import zipfile

# Pack both CSV files into one ZIP
with zipfile.ZipFile(
    "/content/IPL_2027_Eligibility_Files.zip", "w"
) as zipf:
    zipf.write(
        "/content/IPL_2027_Final_Eligible_Players.csv",
        "IPL_2027_Final_Eligible_Players.csv"
    )
    zipf.write(
        "/content/IPL_2027_Ineligible_Players.csv",
        "IPL_2027_Ineligible_Players.csv"
    )

# Download directly to your device
files.download("/content/IPL_2027_Eligibility_Files.zip")


In [ ]:

# EXCLUDE PLAYERS WITH MORE THAN 10 IPL MATCHES

filtered_players = eligible_players[
    eligible_players["ipl_matches_recorded"] <= 10
].copy()

# Sort alphabetically
filtered_players = filtered_players.sort_values(
    "player_name"
).reset_index(drop=True)

# Summary
print("Original Eligible Players:", len(eligible_players))
print("Excluded (More than 10 IPL matches):",
      len(eligible_players) - len(filtered_players))
print("Remaining Players:", len(filtered_players))

display(filtered_players)

# Export final filtered list
filtered_players.to_csv(
    "/content/IPL_2027_Eligible_Under_11_IPL_Matches.csv",
    index=False
)

# Download directly to your device
from google.colab import files

files.download(
    "/content/IPL_2027_Eligible_Under_11_IPL_Matches.csv"
)


In [ ]:
# @title

# STEP 1: Find available player role information

import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        role_cols = [
            col for col in obj.columns
            if any(word in col.lower() for word in [
                "role", "batting", "bowling",
                "position", "style", "player_type"
            ])
        ]

        if role_cols:
            print(f"\nDataset: {name}")
            print("Role columns:", role_cols)
            display(obj[role_cols].head(5))


In [ ]:

# STEP 2 — Inspect player-level data structure

datasets = [
    "filtered_players",
    "eligible_players",
    "eligible_contributions",
    "batting_stats",
    "bowling_stats",
    "playing",
    "balls",
    "player_contributions"
]

for name in datasets:
    if name in globals():
        obj = globals()[name]
        print("\n" + "=" * 55)
        print("DATASET:", name)
        print("Columns:", obj.columns.tolist())
        print("Shape:", obj.shape)
        display(obj.head(3))


In [ ]:

import pandas as pd
import numpy as np
from google.colab import files

# ==========================================
# IPL 2027 — PLAYER ROLE CLASSIFICATION
# ==========================================

# 1. Base candidate pool
pool = filtered_players.copy()

# 2. Attach batting and bowling contributions
pool = pool.merge(
    eligible_contributions[
        ["player_id", "batting_deliveries", "batting_runs",
         "bowling_deliveries", "bowling_matches"]
    ],
    on="player_id",
    how="left"
)

numeric_cols = [
    "batting_deliveries", "batting_runs",
    "bowling_deliveries", "bowling_matches"
]

pool[numeric_cols] = pool[numeric_cols].fillna(0)

# 3. Determine primary batting position
played = playing.copy()

played = played[
    (played["is_played"] == True) &
    (played["batting_order"].notna())
]

position_stats = (
    played.groupby("player_id")["batting_order"]
    .agg(
        primary_batting_position=lambda x: x.mode().iloc[0],
        position_observations="count"
    )
    .reset_index()
)

pool = pool.merge(position_stats, on="player_id", how="left")

# 4. Determine bowling type from actual deliveries
bowling_data = balls.copy()

bowling_data["pace_or_spin"] = (
    bowling_data["pace_or_spin"]
    .astype(str).str.strip().str.lower()
)

bowling_data = bowling_data[
    bowling_data["pace_or_spin"].isin(["pace", "spin"])
]

bowl_type = (
    bowling_data.groupby(["bowler_id", "pace_or_spin"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={
        "bowler_id": "player_id",
        "pace": "pace_balls",
        "spin": "spin_balls"
    })
)

for col in ["pace_balls", "spin_balls"]:
    if col not in bowl_type.columns:
        bowl_type[col] = 0

pool = pool.merge(bowl_type, on="player_id", how="left")

pool[["pace_balls", "spin_balls"]] = (
    pool[["pace_balls", "spin_balls"]].fillna(0)
)

pool["total_type_balls"] = (
    pool["pace_balls"] + pool["spin_balls"]
)

# 5. Define evidence thresholds
MIN_BATTING_BALLS = 60
MIN_BOWLING_BALLS = 60

# 6. Classify players
def classify_player(row):

    bat = row["batting_deliveries"] >= MIN_BATTING_BALLS
    bowl = row["bowling_deliveries"] >= MIN_BOWLING_BALLS

    pace = row["pace_balls"]
    spin = row["spin_balls"]
    total = row["total_type_balls"]

    position = row["primary_batting_position"]

    # All-rounders first
    if bat and bowl:

        if total == 0:
            return "Manual Review"

        if spin > pace:
            return "Spin All-rounder"

        elif pace > spin:
            return "Pace All-rounder"

        else:
            return "Manual Review"

    # Specialist bowlers
    elif bowl and not bat:

        if total == 0:
            return "Manual Review"

        if spin > pace:
            return "Spin Bowler"

        elif pace > spin:
            return "Pace Bowler"

        else:
            return "Manual Review"

    # Specialist batters
    elif bat and not bowl:

        if pd.isna(position):
            return "Manual Review"

        if position <= 3:
            return "Top-order Batter"

        elif position <= 7:
            return "Middle-order Batter"

        else:
            return "Manual Review"

    return "Manual Review"


pool["player_category"] = pool.apply(
    classify_player, axis=1
)

# 7. Create six categories + manual review
categories = [
    "Top-order Batter",
    "Middle-order Batter",
    "Spin All-rounder",
    "Pace All-rounder",
    "Pace Bowler",
    "Spin Bowler"
]

classified = {
    category: pool[
        pool["player_category"] == category
    ].copy()
    for category in categories
}

manual_review = pool[
    pool["player_category"] == "Manual Review"
].copy()

# 8. Summary
print("=" * 60)
print("IPL 2027 — PLAYER ROLE SEGREGATION")
print("=" * 60)

print("Total Candidate Pool:", len(pool))

for category in categories:
    print(f"{category}: {len(classified[category])}")

print("Manual Review:", len(manual_review))

print("\nCATEGORY-WISE PLAYER LISTS")

for category in categories:
    print("\n" + "-" * 45)
    print(category.upper())
    display(
        classified[category][
            ["player_id", "player_name",
             "ipl_matches_recorded",
             "primary_batting_position",
             "batting_deliveries",
             "bowling_deliveries",
             "pace_balls", "spin_balls"]
        ].reset_index(drop=True)
    )

print("\nMANUAL REVIEW")
display(manual_review[
    ["player_id", "player_name", "batting_deliveries",
     "bowling_deliveries", "primary_batting_position"]
])

# 9. Export one Excel workbook
output_file = "/content/IPL_2027_Player_Role_Segregation.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    pool.to_excel(
        writer, sheet_name="All Candidates", index=False
    )

    for category in categories:
        sheet = category.replace("-", "")[:31]
        classified[category].to_excel(
            writer, sheet_name=sheet, index=False
        )

    manual_review.to_excel(
        writer, sheet_name="Manual Review", index=False
    )

print("\nExcel workbook created successfully!")

# 10. Download directly to your device
files.download(output_file)


In [ ]:
manual_audit = manual_review.copy()

manual_audit["batting_deliveries"] = (
    manual_audit["batting_deliveries"].fillna(0)
)

manual_audit["bowling_deliveries"] = (
    manual_audit["bowling_deliveries"].fillna(0)
)

manual_audit["primary_batting_position"] = (
    manual_audit["primary_batting_position"].fillna(0)
)

manual_audit["batting_evidence"] = pd.cut(
    manual_audit["batting_deliveries"],
    bins=[-1, 0, 20, 40, 59, float("inf")],
    labels=["No batting", "1-20", "21-40", "41-59", "60+"]
)

manual_audit["bowling_evidence"] = pd.cut(
    manual_audit["bowling_deliveries"],
    bins=[-1, 0, 20, 40, 59, float("inf")],
    labels=["No bowling", "1-20", "21-40", "41-59", "60+"]
)

print("MANUAL REVIEW — BATTING EVIDENCE")
display(manual_audit["batting_evidence"].value_counts().sort_index())

print("MANUAL REVIEW — BOWLING EVIDENCE")
display(manual_audit["bowling_evidence"].value_counts().sort_index())

print("PLAYERS WITH 20+ BATTING DELIVERIES")
display(
    manual_audit[
        manual_audit["batting_deliveries"] >= 20
    ][[
        "player_id",
        "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "primary_batting_position"
    ]].sort_values("batting_deliveries", ascending=False)
)

In [ ]:
import numpy as np
import pandas as pd

audit = manual_audit.copy()

# Provisional batting candidates
batting_candidates = audit[
    (audit["batting_deliveries"] >= 20) &
    (audit["bowling_deliveries"] < 20)
].copy()

batting_candidates["provisional_role"] = np.where(
    batting_candidates["primary_batting_position"].between(1, 3),
    "Top-order Batter",
    np.where(
        batting_candidates["primary_batting_position"].between(4, 7),
        "Middle-order Batter",
        "Batting Role Review"
    )
)

# Potential all-rounders
allrounder_candidates = audit[
    (audit["batting_deliveries"] >= 20) &
    (audit["bowling_deliveries"] >= 20)
].copy()

allrounder_candidates["provisional_role"] = "Potential All-rounder"

# Potential bowling candidates
bowling_candidates = audit[
    (audit["batting_deliveries"] < 20) &
    (audit["bowling_deliveries"] >= 20)
].copy()

bowling_candidates["provisional_role"] = "Bowling Role Review"

print("PROVISIONAL BATTER CANDIDATES")
display(
    batting_candidates[[
        "player_id", "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "primary_batting_position",
        "provisional_role"
    ]].sort_values("batting_deliveries", ascending=False)
)

print("POTENTIAL ALL-ROUNDERS")
display(
    allrounder_candidates[[
        "player_id", "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "primary_batting_position"
    ]].sort_values("bowling_deliveries", ascending=False)
)

print("BOWLING ROLE REVIEW")
display(
    bowling_candidates[[
        "player_id", "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "primary_batting_position"
    ]].sort_values("bowling_deliveries", ascending=False)
)

In [ ]:
# Combine potential all-rounders and bowling candidates
bowl_audit = pd.concat([
    allrounder_candidates,
    bowling_candidates
]).drop_duplicates(subset="player_id").copy()

# Extract bowling-type data from ball-by-ball records
pace_spin_data = balls.copy()

pace_spin_data["pace_or_spin"] = (
    pace_spin_data["pace_or_spin"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Inspect available bowling-type labels
print("AVAILABLE PACE/SPIN LABELS:")
print(pace_spin_data["pace_or_spin"].value_counts(dropna=False))

# Aggregate pace and spin deliveries player-wise
pace_spin_counts = (
    pace_spin_data[
        pace_spin_data["bowler_id"].isin(bowl_audit["player_id"])
    ]
    .pivot_table(
        index="bowler_id",
        columns="pace_or_spin",
        values="match_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

# Merge pace/spin counts with candidate list
bowl_audit = bowl_audit.merge(
    pace_spin_counts,
    left_on="player_id",
    right_on="bowler_id",
    how="left"
)

# Standardize columns if labels exist
bowl_audit.columns = [
    str(col) for col in bowl_audit.columns
]

print("\nBOWLING CANDIDATES — PACE/SPIN BREAKDOWN")

display(
    bowl_audit[[
        "player_id",
        "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "pace",
        "spin"
    ]].fillna(0).sort_values(
        "bowling_deliveries",
        ascending=False
    )
)

In [ ]:
# -----------------------------------------
# REVISED PROVISIONAL ROLE CLASSIFICATION
# -----------------------------------------

# 1. Batters
batters = batting_candidates.copy()

batters["revised_role"] = np.where(
    batters["primary_batting_position"] <= 3,
    "Top-order Batter",
    "Middle-order Batter"
)

# 2. Bowling-related candidates
bowlers = bowl_audit.copy()

# Ensure pace/spin columns are numeric
bowlers["pace"] = pd.to_numeric(
    bowlers["pace"], errors="coerce"
).fillna(0)

bowlers["spin"] = pd.to_numeric(
    bowlers["spin"], errors="coerce"
).fillna(0)

# Identify all-rounders
bowlers["is_allrounder"] = (
    (bowlers["batting_deliveries"] >= 20) &
    (bowlers["bowling_deliveries"] >= 20)
)

# Assign provisional roles
def assign_role(row):

    if row["is_allrounder"]:
        if row["pace"] > row["spin"]:
            return "Pace All-rounder"
        elif row["spin"] > row["pace"]:
            return "Spin All-rounder"
        else:
            return "All-rounder Review"

    else:
        if row["pace"] > row["spin"]:
            return "Pace Bowler"
        elif row["spin"] > row["pace"]:
            return "Spin Bowler"
        else:
            return "Bowling Review"

bowlers["revised_role"] = bowlers.apply(
    assign_role, axis=1
)

# Combine all 60 candidates
revised_manual_candidates = pd.concat(
    [batters, bowlers],
    ignore_index=True
)

# Summary
print("REVISED PROVISIONAL ROLE COUNTS")
display(
    revised_manual_candidates["revised_role"]
    .value_counts()
)

# Detailed player list
print("\nCOMPLETE REVISED PLAYER LIST")

display(
    revised_manual_candidates[[
        "player_id",
        "player_name",
        "batting_deliveries",
        "bowling_deliveries",
        "primary_batting_position",
        "revised_role"
    ]].sort_values(
        ["revised_role", "player_name"]
    )
)

# Export separately — original workbook remains untouched
revised_manual_candidates.to_excel(
    "/content/IPL_2027_Revised_Manual_Candidates.xlsx",
    index=False
)

print("\nRevised candidate file created successfully!")

In [ ]:
# Correct Ripal Patel's provisional classification

revised_manual_candidates.loc[
    revised_manual_candidates["player_id"] == 8040,
    "revised_role"
] = "Manual Review"

# Updated counts
print("CORRECTED PROVISIONAL ROLE COUNTS")

display(
    revised_manual_candidates["revised_role"]
    .value_counts()
)

# Save corrected file
revised_manual_candidates.to_excel(
    "/content/IPL_2027_Revised_Manual_Candidates.xlsx",
    index=False
)

print("Corrected workbook saved!")

In [ ]:
import pandas as pd

# -----------------------------------------
# 1. LOAD BOTH WORKBOOKS
# -----------------------------------------

original_file = "/content/IPL_2027_Player_Role_Segregation.xlsx"
revised_file = "/content/IPL_2027_Revised_Manual_Candidates.xlsx"

original_all = pd.read_excel(
    original_file,
    sheet_name="All Candidates"
)

revised = pd.read_excel(revised_file)

# -----------------------------------------
# 2. EXTRACT ORIGINAL 62 CLASSIFICATIONS
# -----------------------------------------

role_sheets = {
    "Toporder Batter": "Top-order Batter",
    "Middleorder Batter": "Middle-order Batter",
    "Spin Allrounder": "Spin All-rounder",
    "Pace Allrounder": "Pace All-rounder",
    "Pace Bowler": "Pace Bowler",
    "Spin Bowler": "Spin Bowler"
}

original_role_list = []

for sheet, role in role_sheets.items():

    temp = pd.read_excel(
        original_file,
        sheet_name=sheet
    )

    temp = temp[["player_id"]].copy()
    temp["role_category"] = role
    temp["classification_status"] = "Initial Classification"
    temp["data_confidence"] = "Initial rule-based"

    original_role_list.append(temp)

original_roles = pd.concat(
    original_role_list,
    ignore_index=True
)

# -----------------------------------------
# 3. EXTRACT 59 NEW PROVISIONAL CLASSIFICATIONS
# -----------------------------------------

new_roles = revised[
    revised["revised_role"] != "Manual Review"
][["player_id", "revised_role"]].copy()

new_roles = new_roles.rename(
    columns={"revised_role": "role_category"}
)

new_roles["classification_status"] = "Provisional Classification"
new_roles["data_confidence"] = "Limited delivery sample"

# -----------------------------------------
# 4. COMBINE ROLE ASSIGNMENTS
# -----------------------------------------

all_roles = pd.concat(
    [original_roles, new_roles],
    ignore_index=True
)

# Check duplicate assignments
duplicates = all_roles[
    all_roles.duplicated("player_id", keep=False)
]

print("DUPLICATE PLAYER ASSIGNMENTS:")
display(duplicates)

# -----------------------------------------
# 5. MERGE WITH ALL 155 CANDIDATES
# -----------------------------------------

master = original_all.merge(
    all_roles,
    on="player_id",
    how="left"
)

master["classification_status"] = (
    master["classification_status"].fillna("Manual Review")
)

master["data_confidence"] = (
    master["data_confidence"].fillna("Insufficient Evidence")
)

master["role_category"] = (
    master["role_category"].fillna("Manual Review")
)

# -----------------------------------------
# 6. CREATE SUMMARY
# -----------------------------------------

summary = (
    master.groupby(
        ["role_category", "classification_status"],
        dropna=False
    )
    .size()
    .reset_index(name="player_count")
)

print("\nFINAL ROLE DISTRIBUTION")
display(
    master["role_category"].value_counts()
)

print("\nCLASSIFICATION STATUS")
display(
    master["classification_status"].value_counts()
)

print("\nSUMMARY TABLE")
display(summary)

# -----------------------------------------
# 7. EXPORT FINAL MASTER WORKBOOK
# -----------------------------------------

output_file = "/content/IPL_2027_Master_Player_Role_Database.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    master.to_excel(
        writer,
        sheet_name="Master Database",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Role Summary",
        index=False
    )

    master[
        master["role_category"] == "Manual Review"
    ].to_excel(
        writer,
        sheet_name="Manual Review",
        index=False
    )

    master[
        master["classification_status"] == "Initial Classification"
    ].to_excel(
        writer,
        sheet_name="Initial Classification",
        index=False
    )

    master[
        master["classification_status"] == "Provisional Classification"
    ].to_excel(
        writer,
        sheet_name="Provisional Classification",
        index=False
    )

print("\nMASTER WORKBOOK CREATED SUCCESSFULLY!")
print("Total candidates:", len(master))
print("Classified players:", (master["role_category"] != "Manual Review").sum())
print("Manual Review:", (master["role_category"] == "Manual Review").sum())

from google.colab import files
files.download(output_file)

In [ ]:

import pandas as pd
import numpy as np

# =====================================================
# 1. SETTINGS
# =====================================================

MIN_SAMPLE = 10

# =====================================================
# 2. EXTRACT THE 34 MANUAL REVIEW PLAYERS
# =====================================================

review_players = master[
    master["role_category"] == "Manual Review"
][["player_id", "player_name"]].copy()

print("Players entering expanded review:", len(review_players))

# =====================================================
# 3. GET BATTING AND BOWLING CONTRIBUTIONS
# =====================================================

contributions = eligible_contributions[
    [
        "player_id",
        "batting_deliveries",
        "bowling_deliveries"
    ]
].drop_duplicates("player_id")

review = review_players.merge(
    contributions,
    on="player_id",
    how="left"
)

review[["batting_deliveries", "bowling_deliveries"]] = (
    review[["batting_deliveries", "bowling_deliveries"]]
    .fillna(0)
)

# =====================================================
# 4. GET PRIMARY BATTING POSITION
# =====================================================

def get_primary_position(series):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if values.empty:
        return np.nan

    modes = values.mode()
    return modes.iloc[0] if not modes.empty else np.nan


positions = (
    playing[playing["is_played"] == 1]
    .groupby("player_id")["batting_order"]
    .apply(get_primary_position)
    .reset_index(name="primary_batting_position")
)

review = review.merge(
    positions,
    on="player_id",
    how="left"
)

# =====================================================
# 5. GET PACE AND SPIN BOWLING EVIDENCE
# =====================================================

pace_spin = (
    balls.groupby(["bowler_id", "pace_or_spin"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"bowler_id": "player_id"})
)

for col in ["pace", "spin"]:
    if col not in pace_spin.columns:
        pace_spin[col] = 0

pace_spin = pace_spin[
    ["player_id", "pace", "spin"]
]

review = review.merge(
    pace_spin,
    on="player_id",
    how="left"
)

review[["pace", "spin"]] = (
    review[["pace", "spin"]].fillna(0)
)

# =====================================================
# 6. ASSIGN ROLES USING AVAILABLE EVIDENCE
# =====================================================

def classify_player(row):

    bat = row["batting_deliveries"]
    bowl = row["bowling_deliveries"]

    position = row["primary_batting_position"]

    pace = row["pace"]
    spin = row["spin"]

    batting_evidence = bat >= MIN_SAMPLE
    bowling_evidence = bowl >= MIN_SAMPLE

    # Insufficient evidence in both disciplines
    if not batting_evidence and not bowling_evidence:
        return "Excluded - Small Sample"

    # Both batting and bowling evidence
    if batting_evidence and bowling_evidence:

        if pace > spin:
            return "Pace All-rounder"

        elif spin > pace:
            return "Spin All-rounder"

        else:
            return "Excluded - Bowling Type Unclear"

    # Batting evidence only
    if batting_evidence and not bowling_evidence:

        if pd.isna(position):
            return "Excluded - Position Unclear"

        if position <= 3:
            return "Top-order Batter"

        elif 4 <= position <= 7:
            return "Middle-order Batter"

        else:
            return "Excluded - Lower-order Batter"

    # Bowling evidence only
    if bowling_evidence and not batting_evidence:

        if pace > spin:
            return "Pace Bowler"

        elif spin > pace:
            return "Spin Bowler"

        else:
            return "Excluded - Bowling Type Unclear"


review["new_role"] = review.apply(
    classify_player,
    axis=1
)

# =====================================================
# 7. ADD CLASSIFICATION STATUS
# =====================================================

review["classification_status"] = np.where(
    review["new_role"].str.startswith("Excluded"),
    "Excluded from Active Pool",
    "Expanded Review Classification"
)

review["data_confidence"] = np.where(
    review["classification_status"] == "Excluded from Active Pool",
    "Insufficient or unclear evidence",
    "Low - directional evidence"
)

# =====================================================
# 8. UPDATE MASTER DATABASE
# =====================================================

expanded_master = master.copy()

review_updates = review.set_index("player_id")

for index, row in expanded_master.iterrows():

    player_id = row["player_id"]

    if player_id in review_updates.index:

        update = review_updates.loc[player_id]

        expanded_master.at[index, "role_category"] = update["new_role"]

        expanded_master.at[index, "classification_status"] = (
            update["classification_status"]
        )

        expanded_master.at[index, "data_confidence"] = (
            update["data_confidence"]
        )

# =====================================================
# 9. CREATE ACTIVE AND EXCLUDED POOLS
# =====================================================

excluded_players = expanded_master[
    expanded_master["role_category"].str.startswith("Excluded")
].copy()

active_scouting_pool = expanded_master[
    ~expanded_master["role_category"].str.startswith("Excluded")
].copy()

# =====================================================
# 10. FINAL DISTRIBUTION
# =====================================================

print("\nEXPANDED REVIEW RESULTS")
display(
    review["new_role"].value_counts()
)

print("\nFINAL ACTIVE ROLE DISTRIBUTION")
display(
    active_scouting_pool["role_category"].value_counts()
)

print("\nFINAL COUNTS")
print("Original pool:", len(expanded_master))
print("Active scouting pool:", len(active_scouting_pool))
print("Excluded players:", len(excluded_players))

# =====================================================
# 11. EXPORT FINAL WORKBOOK
# =====================================================

output_file = "/content/IPL_2027_Expanded_Scouting_Database.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    expanded_master.to_excel(
        writer,
        sheet_name="Complete Database",
        index=False
    )

    active_scouting_pool.to_excel(
        writer,
        sheet_name="Active Scouting Pool",
        index=False
    )

    review.to_excel(
        writer,
        sheet_name="Expanded Review Audit",
        index=False
    )

    excluded_players.to_excel(
        writer,
        sheet_name="Excluded Players",
        index=False
    )

    active_scouting_pool.groupby(
        "role_category"
    ).size().reset_index(
        name="player_count"
    ).to_excel(
        writer,
        sheet_name="Final Role Summary",
        index=False
    )

print("\nEXPANDED SCOUTING DATABASE CREATED!")

from google.colab import files
files.download(output_file)


In [ ]:

import pandas as pd
import numpy as np

# ==========================================
# 1. IDENTIFY THE 13 UNCLEAR PLAYERS
# ==========================================

unclear_players = review[
    review["new_role"] == "Excluded - Bowling Type Unclear"
][["player_id", "player_name"]].copy()

print("Players to investigate:", len(unclear_players))

# ==========================================
# 2. FILTER THEIR BALL-BY-BALL RECORDS
# ==========================================

unclear_balls = balls[
    balls["bowler_id"].isin(unclear_players["player_id"])
].copy()

# ==========================================
# 3. AGGREGATE PACE AND SPIN COUNTS
# ==========================================

pace_spin_audit = (
    unclear_balls
    .pivot_table(
        index="bowler_id",
        columns="pace_or_spin",
        values="innings_ball_num",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
    .rename(columns={"bowler_id": "player_id"})
)

for col in ["pace", "spin"]:
    if col not in pace_spin_audit.columns:
        pace_spin_audit[col] = 0

# ==========================================
# 4. ANALYSE BOWLING TYPE
# ==========================================

def get_distribution(series):
    counts = series.dropna().value_counts()
    return " | ".join(
        f"{name}: {count}" for name, count in counts.items()
    )

type_audit = (
    unclear_balls.groupby("bowler_id")
    .agg(
        bowling_type_evidence=("bowling_type", get_distribution),
        delivery_style_evidence=("delivery_style", get_distribution),
        total_balls=("bowler_id", "size")
    )
    .reset_index()
    .rename(columns={"bowler_id": "player_id"})
)

# ==========================================
# 5. COMBINE AUDIT
# ==========================================

bowling_audit = (
    unclear_players
    .merge(pace_spin_audit, on="player_id", how="left")
    .merge(type_audit, on="player_id", how="left")
)

bowling_audit[["pace", "spin", "total_balls"]] = (
    bowling_audit[["pace", "spin", "total_balls"]]
    .fillna(0)
)

bowling_audit["pace_spin_difference"] = (
    bowling_audit["pace"] - bowling_audit["spin"]
)

bowling_audit["pace_percentage"] = np.where(
    bowling_audit["total_balls"] > 0,
    bowling_audit["pace"] / bowling_audit["total_balls"] * 100,
    0
)

bowling_audit["spin_percentage"] = np.where(
    bowling_audit["total_balls"] > 0,
    bowling_audit["spin"] / bowling_audit["total_balls"] * 100,
    0
)

# ==========================================
# 6. DISPLAY RESULTS
# ==========================================

bowling_audit = bowling_audit.sort_values(
    "total_balls",
    ascending=False
)

display(bowling_audit)

# ==========================================
# 7. EXPORT AUDIT
# ==========================================

bowling_audit.to_excel(
    "/content/13_Unclear_Bowlers_Audit.xlsx",
    index=False
)

print("Bowling audit exported successfully!")


In [ ]:

# ==========================================
# 1. ASSIGN THE 13 PLAYERS
# ==========================================

pace_players = [
    1045, 69806, 28664, 35072, 6285, 8743
]

spin_players = [
    19969, 27391, 5665, 36983, 2531, 37731, 32244
]

# ==========================================
# 2. UPDATE MASTER DATABASE
# ==========================================

expanded_master = expanded_master.copy()

expanded_master.loc[
    expanded_master["player_id"].isin(pace_players),
    "role_category"
] = "Pace Bowler"

expanded_master.loc[
    expanded_master["player_id"].isin(spin_players),
    "role_category"
] = "Spin Bowler"

updated_ids = pace_players + spin_players

expanded_master.loc[
    expanded_master["player_id"].isin(updated_ids),
    "classification_status"
] = "Expanded Review - Bowling Style Evidence"

expanded_master.loc[
    expanded_master["player_id"].isin(updated_ids),
    "data_confidence"
] = "Low - 12 to 18 recorded deliveries"

# ==========================================
# 3. REBUILD ACTIVE AND EXCLUDED POOLS
# ==========================================

VALID_ROLES = [
    "Top-order Batter",
    "Middle-order Batter",
    "Spin All-rounder",
    "Pace All-rounder",
    "Pace Bowler",
    "Spin Bowler"
]

active_scouting_pool = expanded_master[
    expanded_master["role_category"].isin(VALID_ROLES)
].copy()

excluded_players = expanded_master[
    ~expanded_master["role_category"].isin(VALID_ROLES)
].copy()

# ==========================================
# 4. FINAL DISTRIBUTION
# ==========================================

print("UPDATED ROLE DISTRIBUTION")

display(
    active_scouting_pool["role_category"].value_counts()
)

print("\nFINAL COUNTS")

print("Original pool:", len(expanded_master))
print("Active scouting pool:", len(active_scouting_pool))
print("Excluded players:", len(excluded_players))

print("\nNEWLY ASSIGNED PLAYERS")

display(
    expanded_master[
        expanded_master["player_id"].isin(updated_ids)
    ][
        ["player_id", "player_name", "role_category",
         "classification_status", "data_confidence"]
    ]
)

# ==========================================
# 5. EXPORT FINAL WORKBOOK
# ==========================================

output_file = "/content/IPL_2027_Final_Expanded_Scouting_Database.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    expanded_master.to_excel(
        writer,
        sheet_name="Complete Database",
        index=False
    )

    active_scouting_pool.to_excel(
        writer,
        sheet_name="Active Scouting Pool",
        index=False
    )

    excluded_players.to_excel(
        writer,
        sheet_name="Excluded Players",
        index=False
    )

    active_scouting_pool.groupby(
        "role_category"
    ).size().reset_index(
        name="player_count"
    ).to_excel(
        writer,
        sheet_name="Final Role Summary",
        index=False
    )

    bowling_audit.to_excel(
        writer,
        sheet_name="Bowling Style Audit",
        index=False
    )

print("\nFINAL EXPANDED SCOUTING DATABASE CREATED!")

from google.colab import files
files.download(output_file)


In [ ]:

import pandas as pd
import numpy as np

# ==========================================
# 1. GET PRIMARY BATTING POSITION
# ==========================================

def get_primary_position(series):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if values.empty:
        return np.nan

    modes = values.mode()
    return modes.iloc[0]

positions = (
    playing[playing["is_played"] == 1]
    .groupby("player_id")["batting_order"]
    .apply(get_primary_position)
    .reset_index(name="primary_batting_position")
)

# Merge positions into active pool
pool = active_scouting_pool.drop(
    columns=["primary_batting_position"],
    errors="ignore"
).merge(
    positions,
    on="player_id",
    how="left"
)

# ==========================================
# 2. IDENTIFY ORIGINAL PLAYER GROUPS
# ==========================================

batter_roles = [
    "Top-order Batter",
    "Middle-order Batter"
]

bowler_roles = [
    "Pace Bowler",
    "Spin Bowler"
]

allrounder_roles = [
    "Pace All-rounder",
    "Spin All-rounder"
]

regular_batters = pool[
    pool["role_category"].isin(batter_roles)
].copy()

regular_bowlers = pool[
    pool["role_category"].isin(bowler_roles)
].copy()

allrounders = pool[
    pool["role_category"].isin(allrounder_roles)
].copy()

# ==========================================
# 3. CLASSIFY ALL-ROUNDERS BY BATTING POSITION
# ==========================================

allrounders["recruitment_category"] = np.select(
    [
        allrounders["primary_batting_position"].between(1, 6),
        allrounders["primary_batting_position"].ge(7)
    ],
    [
        "Batting Pool",
        "Bowling Pool"
    ],
    default="Position Unclear"
)

batting_allrounders = allrounders[
    allrounders["recruitment_category"] == "Batting Pool"
].copy()

bowling_allrounders = allrounders[
    allrounders["recruitment_category"] == "Bowling Pool"
].copy()

unresolved_allrounders = allrounders[
    allrounders["recruitment_category"] == "Position Unclear"
].copy()

# ==========================================
# 4. CREATE FINAL BATTING AND BOWLING POOLS
# ==========================================

batters = pd.concat(
    [regular_batters, batting_allrounders],
    ignore_index=True
)

bowlers = pd.concat(
    [regular_bowlers, bowling_allrounders],
    ignore_index=True
)

# ==========================================
# 5. CREATE ROLE-WISE LISTS
# ==========================================

top_order_batters = batters[
    batters["role_category"] == "Top-order Batter"
].copy()

middle_order_batters = batters[
    batters["role_category"] == "Middle-order Batter"
].copy()

batting_allrounder_list = batters[
    batters["role_category"].isin(allrounder_roles)
].copy()

pace_bowlers = bowlers[
    bowlers["role_category"] == "Pace Bowler"
].copy()

spin_bowlers = bowlers[
    bowlers["role_category"] == "Spin Bowler"
].copy()

bowling_allrounder_list = bowlers[
    bowlers["role_category"].isin(allrounder_roles)
].copy()

# ==========================================
# 6. DISPLAY FINAL COUNTS
# ==========================================

print("FINAL BATTING POOL:", len(batters))
print("Regular batters:", len(regular_batters))
print("Batting all-rounders:", len(batting_allrounders))

print("\nFINAL BOWLING POOL:", len(bowlers))
print("Regular bowlers:", len(regular_bowlers))
print("Bowling all-rounders:", len(bowling_allrounders))

print("\nALL-ROUNDER POSITION UNCLEAR:", len(unresolved_allrounders))

print("\nALL-ROUNDER RECLASSIFICATION")
display(
    allrounders[
        [
            "player_id",
            "player_name",
            "role_category",
            "primary_batting_position",
            "recruitment_category"
        ]
    ].sort_values("primary_batting_position")
)

print("\nBATTING POOL ROLE DISTRIBUTION")
display(batters["role_category"].value_counts())

print("\nBOWLING POOL ROLE DISTRIBUTION")
display(bowlers["role_category"].value_counts())

# ==========================================
# 7. EXPORT RECRUITMENT WORKBOOK
# ==========================================

output_file = "/content/IPL_2027_Batting_Bowling_Recruitment_Pools.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    batters.to_excel(
        writer,
        sheet_name="Batting Pool",
        index=False
    )

    bowlers.to_excel(
        writer,
        sheet_name="Bowling Pool",
        index=False
    )

    allrounders.to_excel(
        writer,
        sheet_name="All-rounder Audit",
        index=False
    )

    top_order_batters.to_excel(
        writer,
        sheet_name="Top-order Batters",
        index=False
    )

    middle_order_batters.to_excel(
        writer,
        sheet_name="Middle-order Batters",
        index=False
    )

    batting_allrounder_list.to_excel(
        writer,
        sheet_name="Batting All-rounders",
        index=False
    )

    pace_bowlers.to_excel(
        writer,
        sheet_name="Pace Bowlers",
        index=False
    )

    spin_bowlers.to_excel(
        writer,
        sheet_name="Spin Bowlers",
        index=False
    )

    bowling_allrounder_list.to_excel(
        writer,
        sheet_name="Bowling All-rounders",
        index=False
    )

    unresolved_allrounders.to_excel(
        writer,
        sheet_name="Position Unclear",
        index=False
    )

print("\nRECRUITMENT POOLS CREATED SUCCESSFULLY!")

from google.colab import files
files.download(output_file)


In [ ]:
from google.colab import files

# Create separate batting workbook
batting_file = "/content/IPL_2027_Batting_Recruitment_Pool.xlsx"

with pd.ExcelWriter(batting_file, engine="openpyxl") as writer:
    batters.to_excel(writer, sheet_name="Batting Pool", index=False)
    top_order_batters.to_excel(writer, sheet_name="Top-order Batters", index=False)
    middle_order_batters.to_excel(writer, sheet_name="Middle-order Batters", index=False)
    batting_allrounder_list.to_excel(writer, sheet_name="Batting All-rounders", index=False)

# Create separate bowling workbook
bowling_file = "/content/IPL_2027_Bowling_Recruitment_Pool.xlsx"

with pd.ExcelWriter(bowling_file, engine="openpyxl") as writer:
    bowlers.to_excel(writer, sheet_name="Bowling Pool", index=False)
    pace_bowlers.to_excel(writer, sheet_name="Pace Bowlers", index=False)
    spin_bowlers.to_excel(writer, sheet_name="Spin Bowlers", index=False)
    bowling_allrounder_list.to_excel(writer, sheet_name="Bowling All-rounders", index=False)

# Download both separately
files.download(batting_file)
files.download(bowling_file)

In [ ]:
from google.colab import files
import pandas as pd

# Upload Excel file
uploaded = files.upload()

# Get uploaded filename
file_name = next(iter(uploaded))

# Read Excel workbook
df = pd.read_excel(file_name, sheet_name=0)

# Display results
print("Excel dataset uploaded successfully!")
print("File:", file_name)
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

display(df.head())

In [ ]:
import pandas as pd
import numpy as np

# Read uploaded Excel workbook
excel_file = pd.ExcelFile(file_name)

print("AVAILABLE SHEETS:")
for i, sheet in enumerate(excel_file.sheet_names, 1):
    print(f"{i}. {sheet}")

# Load first sheet
df = pd.read_excel(file_name, sheet_name=0)

# Clean column names
df.columns = df.columns.astype(str).str.strip()

print("\nDATASET OVERVIEW")
print("-" * 50)
print("Total Players:", len(df))
print("Total Columns:", len(df.columns))

print("\nCOLUMN NAMES:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nFIRST 5 ROWS:")
display(df.head())

print("\nMISSING VALUES:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDATA TYPES:")
display(df.dtypes.to_frame("Data Type"))

In [ ]:

import pandas as pd
import numpy as np

# ==========================================
# IPL 2027 - BATTING SCOUTING MODEL
# 25 CRITERIA - VERSION 1
# ==========================================

df = df.copy()

# ------------------------------------------
# 1. DATA CLEANING
# ------------------------------------------

numeric_cols = [
    "Age", "Matches", "Total Runs",
    "Balls vs Pace", "SR vs Pace",
    "Balls vs Spin", "SR vs Spin",
    "Runs (Bat 1st)", "SR (Bat 1st)",
    "Runs (Chasing)", "SR (Chasing)"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Major Position"] = (
    df["Major Position"].astype(str).str.extract(r"(\d+)")[0]
)

df["Major Position"] = pd.to_numeric(
    df["Major Position"], errors="coerce"
)

# ------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------

def percentile_score(series):
    """Convert values into 0-100 percentile scores."""
    values = pd.to_numeric(series, errors="coerce")
    return values.rank(pct=True) * 100


def sample_confidence(balls, minimum=30, strong=150):
    """
    Confidence based on balls faced.
    Low sample = lower confidence.
    """
    balls = pd.to_numeric(balls, errors="coerce")

    confidence = (
        (balls - minimum) / (strong - minimum)
    ).clip(0, 1)

    return confidence.fillna(0)


def age_score(age):
    """Provisional age suitability score."""
    if pd.isna(age):
        return 50
    elif 23 <= age <= 28:
        return 100
    elif 21 <= age < 23 or 29 <= age <= 30:
        return 85
    elif 19 <= age < 21 or 31 <= age <= 32:
        return 65
    else:
        return 40


# ------------------------------------------
# 3. DERIVED PERFORMANCE METRICS
# ------------------------------------------

df["Runs Per Match"] = (
    df["Total Runs"] / df["Matches"].replace(0, np.nan)
)

df["Bat First Runs Per Match"] = (
    df["Runs (Bat 1st)"] / df["Matches"].replace(0, np.nan)
)

df["Chasing Runs Per Match"] = (
    df["Runs (Chasing)"] / df["Matches"].replace(0, np.nan)
)

df["Pace Confidence"] = sample_confidence(
    df["Balls vs Pace"]
)

df["Spin Confidence"] = sample_confidence(
    df["Balls vs Spin"]
)

# Sample-adjusted matchup performance.
# 50 is neutral when there is no reliable sample.

df["Pace Adjusted SR"] = (
    50 + (df["SR vs Pace"].fillna(50) - 50)
    * df["Pace Confidence"]
)

df["Spin Adjusted SR"] = (
    50 + (df["SR vs Spin"].fillna(50) - 50)
    * df["Spin Confidence"]
)

df["Pace Spin Balance"] = (
    100 - (
        df["SR vs Pace"] - df["SR vs Spin"]
    ).abs().fillna(0)
)

df["First Chase Balance"] = (
    100 - (
        df["SR (Bat 1st)"] - df["SR (Chasing)"]
    ).abs().fillna(0)
)

df["Overall SR Profile"] = df[
    ["SR vs Pace", "SR vs Spin",
     "SR (Bat 1st)", "SR (Chasing)"]
].mean(axis=1)

# ------------------------------------------
# 4. POSITION-BASED SUITABILITY
# ------------------------------------------

def top_order_score(row):
    return np.mean([
        row["SR vs Pace"],
        row["SR (Bat 1st)"],
        row["SR (Chasing)"]
    ])

def middle_order_score(row):
    return np.mean([
        row["SR vs Spin"],
        row["SR (Chasing)"],
        row["SR vs Pace"]
    ])

def lower_order_score(row):
    return np.mean([
        row["SR vs Pace"],
        row["SR (Chasing)"],
        row["SR vs Spin"]
    ])

df["Top Order Raw"] = df.apply(top_order_score, axis=1)
df["Middle Order Raw"] = df.apply(middle_order_score, axis=1)
df["Lower Order Raw"] = df.apply(lower_order_score, axis=1)

df["Position Suitability Raw"] = np.select(
    [
        df["Major Position"].between(1, 2),
        df["Major Position"].between(3, 5),
        df["Major Position"].ge(6)
    ],
    [
        df["Top Order Raw"],
        df["Middle Order Raw"],
        df["Lower Order Raw"]
    ],
    default=50
)

# ------------------------------------------
# 5. CREATE 25 CRITERION SCORES
# ------------------------------------------

criteria = pd.DataFrame(index=df.index)

# 1-4: Experience and production
criteria["C01_Age_Profile"] = df["Age"].apply(age_score)

criteria["C02_Experience"] = percentile_score(df["Matches"])

criteria["C03_Career_Runs"] = percentile_score(df["Total Runs"])

criteria["C04_Runs_Per_Match"] = percentile_score(
    df["Runs Per Match"]
)

# 5-8: Matchup performance and sample
criteria["C05_Pace_SR"] = percentile_score(df["SR vs Pace"])

criteria["C06_Spin_SR"] = percentile_score(df["SR vs Spin"])

criteria["C07_Pace_Sample"] = percentile_score(
    df["Balls vs Pace"]
)

criteria["C08_Spin_Sample"] = percentile_score(
    df["Balls vs Spin"]
)

# 9-14: Innings performance
criteria["C09_Bat_First_SR"] = percentile_score(
    df["SR (Bat 1st)"]
)

criteria["C10_Chasing_SR"] = percentile_score(
    df["SR (Chasing)"]
)

criteria["C11_Bat_First_Runs"] = percentile_score(
    df["Runs (Bat 1st)"]
)

criteria["C12_Chasing_Runs"] = percentile_score(
    df["Runs (Chasing)"]
)

criteria["C13_First_Runs_Per_Match"] = percentile_score(
    df["Bat First Runs Per Match"]
)

criteria["C14_Chase_Runs_Per_Match"] = percentile_score(
    df["Chasing Runs Per Match"]
)

# 15-19: Advanced derived metrics
criteria["C15_Pace_Adjusted"] = percentile_score(
    df["Pace Adjusted SR"]
)

criteria["C16_Spin_Adjusted"] = percentile_score(
    df["Spin Adjusted SR"]
)

criteria["C17_Matchup_Balance"] = percentile_score(
    df["Pace Spin Balance"]
)

criteria["C18_Innings_Balance"] = percentile_score(
    df["First Chase Balance"]
)

criteria["C19_Overall_SR"] = percentile_score(
    df["Overall SR Profile"]
)

# 20: Actual position suitability
criteria["C20_Position_Suitability"] = percentile_score(
    df["Position Suitability Raw"]
)

# 21-23: Role-specific suitability
criteria["C21_Top_Order_Suitability"] = percentile_score(
    df["Top Order Raw"]
)

criteria["C22_Middle_Order_Suitability"] = percentile_score(
    df["Middle Order Raw"]
)

criteria["C23_Lower_Order_Suitability"] = percentile_score(
    df["Lower Order Raw"]
)

# 24-25: Squad flexibility
criteria["C24_Wicketkeeping_Value"] = (
    df["WK Option"].str.strip().str.lower()
    .eq("yes").astype(int) * 100
)

criteria["C25_Bowling_Flexibility"] = (
    df["Bowling Option"].str.strip().str.lower()
    .ne("no").astype(int) * 100
)

# ------------------------------------------
# 6. FINAL CRITERIA SCORE
# ------------------------------------------

# Equal weighting for initial testing.
# We will calibrate weights after reviewing results.

df["Total_Criteria_Score"] = criteria.mean(axis=1)

# Attach individual criterion scores
df = pd.concat([df, criteria], axis=1)

# Rank players
df["Scouting_Rank"] = (
    df["Total_Criteria_Score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# ------------------------------------------
# 7. OUTPUT
# ------------------------------------------

df = df.sort_values(
    "Total_Criteria_Score",
    ascending=False
).reset_index(drop=True)

print("25-CRITERION SCOUTING MODEL COMPLETED")
print("=" * 55)

display(
    df[
        [
            "Scouting_Rank",
            "Player Name",
            "Major Position",
            "Total_Criteria_Score"
        ]
    ].head(20)
)

print("\nIndividual Criterion Scores:")
display(
    df[
        ["Player Name"] + list(criteria.columns)
    ].head(10)
)

# Export workbook
output_file = "/content/RR_25_Criteria_Batting_Model.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Complete Scouting Scores",
        index=False
    )

    df.head(25).to_excel(
        writer,
        sheet_name="Top 25 Shortlist",
        index=False
    )

    criteria.describe().to_excel(
        writer,
        sheet_name="Criteria Statistics"
    )

print("\nWorkbook created:", output_file)


In [ ]:

# FIX 1: Missing sample-size scores
# Missing data is treated as neutral, not zero.

sample_columns = [
    "C07_Pace_Sample",
    "C08_Spin_Sample"
]

for col in sample_columns:
    df[col] = df[col].fillna(50)

# FIX 2: Recalculate final score
# Temporary equal-weight score for diagnostic purposes.

criterion_columns = [
    col for col in df.columns
    if col.startswith("C") and "_" in col
    and col[1:3].isdigit()
]

df["Total_Criteria_Score"] = (
    df[criterion_columns].mean(axis=1)
)

df["Scouting_Rank"] = (
    df["Total_Criteria_Score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

df = df.sort_values(
    "Total_Criteria_Score",
    ascending=False
).reset_index(drop=True)

print("UPDATED SCOUTING RANKING")
display(
    df[
        [
            "Scouting_Rank",
            "Player Name",
            "Major Position",
            "Total_Criteria_Score"
        ]
    ].head(20)
)


In [ ]:

# ==========================================
# IPL 2027 - ROLE SPECIFIC BATTING MODEL
# ==========================================

# Use the existing 25 criterion scores.
role_df = df.copy()

# ------------------------------------------
# 1. DEFINE ROLE-SPECIFIC WEIGHTS
# ------------------------------------------

role_weights = {

    "Top Order": {
        "C05_Pace_SR": 0.15,
        "C06_Spin_SR": 0.08,
        "C09_Bat_First_SR": 0.15,
        "C10_Chasing_SR": 0.12,
        "C04_Runs_Per_Match": 0.12,
        "C15_Pace_Adjusted": 0.12,
        "C16_Spin_Adjusted": 0.06,
        "C17_Matchup_Balance": 0.05,
        "C18_Innings_Balance": 0.05,
        "C02_Experience": 0.05,
        "C03_Career_Runs": 0.05
    },

    "Middle Order": {
        "C05_Pace_SR": 0.08,
        "C06_Spin_SR": 0.15,
        "C09_Bat_First_SR": 0.08,
        "C10_Chasing_SR": 0.15,
        "C04_Runs_Per_Match": 0.12,
        "C15_Pace_Adjusted": 0.06,
        "C16_Spin_Adjusted": 0.12,
        "C17_Matchup_Balance": 0.08,
        "C18_Innings_Balance": 0.08,
        "C02_Experience": 0.04,
        "C03_Career_Runs": 0.04
    },

    "Lower Order": {
        "C05_Pace_SR": 0.15,
        "C06_Spin_SR": 0.08,
        "C09_Bat_First_SR": 0.08,
        "C10_Chasing_SR": 0.15,
        "C04_Runs_Per_Match": 0.10,
        "C15_Pace_Adjusted": 0.12,
        "C16_Spin_Adjusted": 0.06,
        "C17_Matchup_Balance": 0.06,
        "C18_Innings_Balance": 0.10,
        "C02_Experience": 0.05,
        "C03_Career_Runs": 0.05
    }
}

# ------------------------------------------
# 2. VALIDATE WEIGHTS
# ------------------------------------------

for role, weights in role_weights.items():
    assert abs(sum(weights.values()) - 1.0) < 0.001, (
        f"{role} weights do not total 100%"
    )

# ------------------------------------------
# 3. CALCULATE ROLE SCORES
# ------------------------------------------

def calculate_role_score(row, weights):
    return sum(
        row[criterion] * weight
        for criterion, weight in weights.items()
    )

for role, weights in role_weights.items():
    role_df[f"{role} Score"] = role_df.apply(
        lambda row: calculate_role_score(row, weights),
        axis=1
    )

# ------------------------------------------
# 4. CREATE POSITION GROUPS
# ------------------------------------------

role_df["Recruitment Category"] = np.select(
    [
        role_df["Major Position"].between(1, 2),
        role_df["Major Position"].between(3, 5),
        role_df["Major Position"].ge(6)
    ],
    [
        "Top Order",
        "Middle Order",
        "Lower Order"
    ],
    default="Unclassified"
)

# ------------------------------------------
# 5. GENERATE SEPARATE RANKINGS
# ------------------------------------------

top_order_df = (
    role_df[role_df["Recruitment Category"] == "Top Order"]
    .sort_values("Top Order Score", ascending=False)
    .copy()
)

middle_order_df = (
    role_df[role_df["Recruitment Category"] == "Middle Order"]
    .sort_values("Middle Order Score", ascending=False)
    .copy()
)

lower_order_df = (
    role_df[role_df["Recruitment Category"] == "Lower Order"]
    .sort_values("Lower Order Score", ascending=False)
    .copy()
)

top_order_df["Role Rank"] = range(1, len(top_order_df) + 1)
middle_order_df["Role Rank"] = range(1, len(middle_order_df) + 1)
lower_order_df["Role Rank"] = range(1, len(lower_order_df) + 1)

# ------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------

display_columns = [
    "Role Rank",
    "Player Name",
    "Major Position",
    "Age",
    "Total Runs",
    "Matches"
]

print("\nTOP ORDER RANKING")
display(
    top_order_df[
        display_columns + ["Top Order Score"]
    ].head(15)
)

print("\nMIDDLE ORDER RANKING")
display(
    middle_order_df[
        display_columns + ["Middle Order Score"]
    ].head(15)
)

print("\nLOWER ORDER RANKING")
display(
    lower_order_df[
        display_columns + ["Lower Order Score"]
    ].head(15)
)

# ------------------------------------------
# 7. EXPORT ROLE-SPECIFIC WORKBOOK
# ------------------------------------------

output_file = "/content/RR_Role_Specific_Batting_Scouting.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    top_order_df.to_excel(
        writer, sheet_name="Top Order Ranking", index=False
    )

    middle_order_df.to_excel(
        writer, sheet_name="Middle Order Ranking", index=False
    )

    lower_order_df.to_excel(
        writer, sheet_name="Lower Order Ranking", index=False
    )

    role_df.to_excel(
        writer, sheet_name="All Players", index=False
    )

print("\nROLE-SPECIFIC WORKBOOK CREATED:", output_file)


In [ ]:

# ==========================================
# IPL 2027 - FINAL TOP 10 BATTING SELECTION
# Composition: 4 Middle + 4 Lower + 2 Top
# Pakistan Players Excluded
# ==========================================

# 1. Start with role-specific dataset
selection_pool = role_df.copy()

# 2. Remove Pakistan players
selection_pool = selection_pool[
    ~selection_pool["Nationality"]
    .astype(str)
    .str.contains("pakistan", case=False, na=False)
].copy()

# 3. Define exact selection sequence
selection_slots = [
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score")
]

# 4. Select highest-ranked available player per slot
selected_players = []
selected_names = set()

for slot_number, (required_role, score_column) in enumerate(
    selection_slots, start=1
):

    candidates = selection_pool[
        (selection_pool["Recruitment Category"] == required_role) &
        (~selection_pool["Player Name"].isin(selected_names))
    ].sort_values(
        score_column,
        ascending=False
    )

    if candidates.empty:
        print(f"WARNING: Slot {slot_number} ({required_role}) could not be filled.")
        continue

    selected = candidates.iloc[0].copy()

    selected["Final Slot"] = slot_number
    selected["Selected Role"] = required_role
    selected["Selection Score"] = selected[score_column]

    selected_players.append(selected)
    selected_names.add(selected["Player Name"])

# 5. Create final Top 10 dataframe
final_top10 = pd.DataFrame(selected_players)

final_top10 = final_top10.sort_values(
    "Final Slot"
).reset_index(drop=True)

# 6. Arrange presentation columns
final_columns = [
    "Final Slot",
    "Player Name",
    "Nationality",
    "Age",
    "Major Position",
    "Selected Role",
    "Selection Score",
    "Matches",
    "Total Runs",
    "WK Option",
    "Bowling Option"
]

final_top10 = final_top10[final_columns]

# 7. Validate squad composition
print("\nFINAL TOP 10 BATTING RECRUITMENT LIST")
print("=" * 60)

display(final_top10)

print("\nROLE COMPOSITION:")
display(
    final_top10["Selected Role"]
    .value_counts()
    .reindex(
        ["Middle Order", "Lower Order", "Top Order"],
        fill_value=0
    ).to_frame("Selected Players")
)

print("\nTOTAL SELECTED:", len(final_top10))
print(
    "PAKISTAN PLAYERS:",
    final_top10["Nationality"]
    .astype(str)
    .str.contains("pakistan", case=False, na=False)
    .sum()
)

# 8. Export final workbook
output_file = "/content/RR_IPL_2027_Final_Top_10_Batters.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    final_top10.to_excel(
        writer,
        sheet_name="Final Top 10",
        index=False
    )

    selection_pool.to_excel(
        writer,
        sheet_name="Eligible Non-Pakistan Pool",
        index=False
    )

print("\nFINAL WORKBOOK CREATED:", output_file)

# Download in Google Colab
from google.colab import files
files.download(output_file)


In [ ]:

# Remove Pakistan and Bangladesh players

excluded_nationalities = ["pakistan", "bangladesh"]

selection_pool = role_df[
    ~role_df["Nationality"]
    .astype(str)
    .str.lower()
    .str.contains("|".join(excluded_nationalities), na=False)
].copy()

print("Eligible Players Remaining:", len(selection_pool))
print("Excluded Nationalities:", excluded_nationalities)


In [ ]:

# ==========================================
# FINAL TOP 10 - IPL 2027 BATTING
# Pakistan & Bangladesh Excluded
# Composition: 4 Middle + 4 Lower + 2 Top
# ==========================================

# Selection order
selection_slots = [
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score")
]

selected_players = []
selected_names = set()

for slot, (role, score_column) in enumerate(selection_slots, start=1):

    candidates = selection_pool[
        (selection_pool["Recruitment Category"] == role) &
        (~selection_pool["Player Name"].isin(selected_names))
    ].sort_values(score_column, ascending=False)

    if candidates.empty:
        print(f"WARNING: Slot {slot} ({role}) could not be filled.")
        continue

    player = candidates.iloc[0].copy()

    player["Final Slot"] = slot
    player["Selected Role"] = role
    player["Selection Score"] = player[score_column]

    selected_players.append(player)
    selected_names.add(player["Player Name"])

# Final dataframe
final_top10 = pd.DataFrame(selected_players)

final_top10 = final_top10.sort_values(
    "Final Slot"
).reset_index(drop=True)

# Display columns
display_columns = [
    "Final Slot",
    "Player Name",
    "Nationality",
    "Age",
    "Major Position",
    "Selected Role",
    "Selection Score",
    "Matches",
    "Total Runs",
    "WK Option",
    "Bowling Option"
]

final_top10 = final_top10[display_columns]

print("\nFINAL IPL 2027 TOP 10 BATTING SHORTLIST")
print("=" * 65)
display(final_top10)

# Composition verification
print("\nFINAL ROLE COMPOSITION:")
display(final_top10["Selected Role"].value_counts())

print("\nTOTAL SELECTED:", len(final_top10))

# Export
output_file = "/content/RR_IPL_2027_Final_Top_10_Batters.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_top10.to_excel(
        writer,
        sheet_name="Final Top 10",
        index=False
    )

print("\nExcel created:", output_file)

from google.colab import files
files.download(output_file)


In [ ]:

# ==========================================
# FINAL ELIGIBILITY FILTER
# ==========================================

# Exclude Pakistan, Bangladesh and players aged above 32

selection_pool = role_df[
    (~role_df["Nationality"].astype(str).str.lower()
     .str.contains("pakistan|bangladesh", na=False))
    &
    (pd.to_numeric(role_df["Age"], errors="coerce") <= 32)
].copy()

print("ELIGIBILITY FILTER RESULTS")
print("-" * 40)
print("Eligible players:", len(selection_pool))
print("Maximum age allowed: 32")
print("Pakistan excluded: Yes")
print("Bangladesh excluded: Yes")
print("Players aged above 32 excluded: Yes")


In [ ]:

# ==========================================
# FINAL TOP 10 BATTING RECRUITMENT LIST
# IPL 2027 - RAJASTHAN ROYALS
# ==========================================

selection_slots = [
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Middle Order", "Middle Order Score"),
    ("Lower Order", "Lower Order Score"),
    ("Top Order", "Top Order Score")
]

selected_players = []
selected_names = set()

for slot, (role, score_column) in enumerate(selection_slots, start=1):

    candidates = selection_pool[
        (selection_pool["Recruitment Category"] == role) &
        (~selection_pool["Player Name"].isin(selected_names))
    ].sort_values(score_column, ascending=False)

    if candidates.empty:
        print(f"WARNING: Slot {slot} ({role}) could not be filled.")
        continue

    player = candidates.iloc[0].copy()
    player["Final Slot"] = slot
    player["Selected Role"] = role
    player["Selection Score"] = player[score_column]

    selected_players.append(player)
    selected_names.add(player["Player Name"])

final_top10 = pd.DataFrame(selected_players)

final_top10 = final_top10.sort_values(
    "Final Slot"
).reset_index(drop=True)

# Final presentation
final_top10 = final_top10[
    [
        "Final Slot",
        "Player Name",
        "Nationality",
        "Age",
        "Major Position",
        "Selected Role",
        "Selection Score",
        "Matches",
        "Total Runs",
        "WK Option",
        "Bowling Option"
    ]
]

print("\nRAJASTHAN ROYALS — FINAL TOP 10 BATTERS")
print("=" * 65)

display(final_top10)

print("\nFINAL COMPOSITION:")
display(
    final_top10["Selected Role"]
    .value_counts()
    .reindex(
        ["Middle Order", "Lower Order", "Top Order"],
        fill_value=0
    )
)

# Validation
assert len(final_top10) == 10, "All 10 slots could not be filled!"
assert final_top10["Player Name"].nunique() == 10
assert (final_top10["Age"] <= 32).all()
assert not final_top10["Nationality"].str.contains(
    "pakistan|bangladesh", case=False, na=False
).any()

print("\nALL FINAL ELIGIBILITY CHECKS PASSED.")

# Export
output_file = "/content/RR_IPL_2027_Final_Top_10_Batters.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_top10.to_excel(
        writer,
        sheet_name="Final Top 10",
        index=False
    )

print("Workbook saved:", output_file)

from google.colab import files
files.download(output_file)


In [ ]:
from google.colab import files
files.download("/content/RR_IPL_2027_Final_Top_10_Batters.xlsx")

In [ ]:

import pandas as pd
import numpy as np

# ==========================================
# IPL 2027 - FINAL TOP 10 SCOUTING REPORT
# ==========================================

report = final_top10.copy()

# Retrieve complete player data and all 25 criterion scores
player_details = role_df.drop_duplicates(
    subset=["Player Name"]
).copy()

report = report.merge(
    player_details,
    on="Player Name",
    how="left",
    suffixes=("", "_Detail")
)

# ------------------------------------------
# 1. ROLE-SPECIFIC SCORING BREAKDOWN
# ------------------------------------------

def get_role_score_column(role):
    return {
        "Top Order": "Top Order Score",
        "Middle Order": "Middle Order Score",
        "Lower Order": "Lower Order Score"
    }.get(role)

report["Role Score"] = report.apply(
    lambda r: r[get_role_score_column(r["Selected Role"])],
    axis=1
)

# ------------------------------------------
# 2. GENERATE PLAYER SELECTION REASONS
# ------------------------------------------

def selection_reason(row):

    role = row["Selected Role"]
    weights = role_weights[role]

    contributions = []

    for criterion, weight in weights.items():
        score = row.get(criterion, np.nan)

        if pd.notna(score):
            contributions.append({
                "criterion": criterion,
                "score": float(score),
                "contribution": float(score) * weight
            })

    contributions.sort(
        key=lambda x: x["contribution"],
        reverse=True
    )

    top_strengths = contributions[:3]

    strength_names = {
        "C05_Pace_SR": "Pace strike-rate performance",
        "C06_Spin_SR": "Spin strike-rate performance",
        "C09_Bat_First_SR": "Batting-first performance",
        "C10_Chasing_SR": "Chasing performance",
        "C04_Runs_Per_Match": "Runs-per-match output",
        "C15_Pace_Adjusted": "Sample-adjusted pace performance",
        "C16_Spin_Adjusted": "Sample-adjusted spin performance",
        "C17_Matchup_Balance": "Pace-spin matchup balance",
        "C18_Innings_Balance": "Batting-first and chasing balance",
        "C02_Experience": "T20 match experience",
        "C03_Career_Runs": "Career run production"
    }

    reasons = [
        strength_names.get(x["criterion"], x["criterion"])
        for x in top_strengths
    ]

    return "; ".join(reasons)


report["Key Selection Reasons"] = report.apply(
    selection_reason,
    axis=1
)

# ------------------------------------------
# 3. ROLE-SPECIFIC SELECTION EXPLANATION
# ------------------------------------------

role_explanations = {
    "Top Order":
        "Selected for a top-order role based on the highest "
        "available Top Order Score among eligible candidates.",

    "Middle Order":
        "Selected for a middle-order role based on the highest "
        "available Middle Order Score among eligible candidates.",

    "Lower Order":
        "Selected for a lower-order role based on the highest "
        "available Lower Order Score among eligible candidates."
}

report["Why Selected"] = report["Selected Role"].map(
    role_explanations
)

# ------------------------------------------
# 4. ELIGIBILITY AND SQUAD VALUE
# ------------------------------------------

def squad_value(row):

    values = []

    if str(row.get("WK Option", "")).strip().lower() == "yes":
        values.append("Wicketkeeping option")

    bowling = str(row.get("Bowling Option", "")).strip().lower()

    if bowling not in ["no", "", "nan"]:
        values.append("Additional bowling option")

    if not values:
        return "Specialist batting option"

    return " + ".join(values)


report["Squad Flexibility"] = report.apply(
    squad_value, axis=1
)

report["Eligibility Status"] = "Eligible"

# ------------------------------------------
# 5. FINAL PRESENTATION TABLE
# ------------------------------------------

presentation_columns = [
    "Final Slot",
    "Player Name",
    "Nationality",
    "Age",
    "Selected Role",
    "Role Score",
    "Key Selection Reasons",
    "Why Selected",
    "Squad Flexibility",
    "Matches",
    "Total Runs",
    "SR vs Pace",
    "SR vs Spin",
    "SR (Bat 1st)",
    "SR (Chasing)"
]

presentation = report[
    [c for c in presentation_columns if c in report.columns]
].copy()

print("FINAL TOP 10 - PLAYER SELECTION RATIONALE")
display(presentation)

# ------------------------------------------
# 6. COMPLETE 25-CRITERIA SCORECARD
# ------------------------------------------

criterion_columns = [
    c for c in report.columns
    if c.startswith("C") and c[1:3].isdigit()
]

scorecard_columns = [
    "Final Slot",
    "Player Name",
    "Nationality",
    "Age",
    "Selected Role",
    "Role Score"
] + criterion_columns

scorecard = report[
    [c for c in scorecard_columns if c in report.columns]
].copy()

# ------------------------------------------
# 7. EXPORT FINAL SCOUTING WORKBOOK
# ------------------------------------------

output_file = "/content/RR_IPL_2027_Final_Top_10_Detailed_Scouting.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    presentation.to_excel(
        writer,
        sheet_name="Final Top 10",
        index=False
    )

    scorecard.to_excel(
        writer,
        sheet_name="25 Criteria Scorecard",
        index=False
    )

    report.to_excel(
        writer,
        sheet_name="Complete Selection Breakdown",
        index=False
    )

print("\nFINAL SCOUTING WORKBOOK CREATED")
print("Selected Players:", len(report))
print("Criteria Included:", len(criterion_columns))
print("Output:", output_file)

from google.colab import files
files.download(output_file)


In [ ]:
import pandas as pd
from google.colab import files

# Upload your XLSX file
uploaded = files.upload()

# Read uploaded Excel file
file_name = next(iter(uploaded))

df = pd.read_excel(file_name)

# Display the complete dataset
print("File uploaded successfully!")
print("Total Players:", len(df))
print("Columns:", list(df.columns))

display(df)

Saving cricket_player_performance.xlsx to cricket_player_performance.xlsx
File uploaded successfully!
Total Players: 35
Columns: ['Player Name', 'Player Type', 'Age', 'Matches', 'Yorker %', 'Death Economy', 'Death Dot Ball %', 'Middle Phase SR', 'Middle Balls / Bndry', 'Tactical Roster Utility']


,Player Name,Player Type,Age,Matches,Yorker %,Death Economy,Death Dot Ball %,Middle Phase SR,Middle Balls / Bndry,Tactical Roster Utility
0,Raj Bawa,Pace AR,24,8,32.0%,9.55,31.2%,15.4,8.1,Ideal Finisher Seam AR - 32 deliveries faced b...
1,Muzahir Raza,Pace Bowler,24,6,31.2%,9.38,31.0%,14.9,8.4,Lower-Order Hitter - 46 runs off 28 balls to s...
2,Krish Bhagat,Pace AR,24,3,22.4%,10.12,25.0%,18.1,6.5,Operational Reserve - Stabilizing depth covera...
3,Chetan Sakariya,Pace Bowler,24,6,33.5%,9.15,31.5%,15.3,8.2,Elite Youth Target - Left arm wide yorker deat...
4,Aaqib Khan,Pace Bowler,21,2,34.0%,9.22,32.0%,14.8,7.9,High-signal Powerplay-to-Death containment pro...
5,Darshan Nalkande,Pace Bowler,24,6,31.0%,9.38,30.0%,14.9,8.4,Exceptional under-25 death overs change-of-pac...
6,Manav Suthar,Spin Bowler,24,5,0.0%,9.12,31.5%,14.0,9.3,Middle Phase Disruptor - Smashes elite 14.2 ba...
7,Musheer Khan,Spin Bowler,21,1,0.0%,9.25,31.0%,14.2,9.1,Attacking wrist-spin option beating playoff me...
8,Rajvardhan Hangargekar,Pace Bowler,24,2,32.0%,9.30,30.5%,15.0,8.1,High-velocity powerplay-death asset triggering...
9,Sushant Mishra,Pace Bowler,24,2,30.5%,9.40,29.8%,15.1,8.0,Left-arm bouncer variation tracking under-25 p...


In [ ]:
print("TOTAL PLAYERS:", len(df))
print("\nAVAILABLE COLUMNS:")
for col in df.columns:
    print("-", col)

display(df.head(5))

TOTAL PLAYERS: 35

AVAILABLE COLUMNS:
- Player Name
- Player Type
- Age
- Matches
- Yorker %
- Death Economy
- Death Dot Ball %
- Middle Phase SR
- Middle Balls / Bndry
- Tactical Roster Utility


,Player Name,Player Type,Age,Matches,Yorker %,Death Economy,Death Dot Ball %,Middle Phase SR,Middle Balls / Bndry,Tactical Roster Utility
0,Raj Bawa,Pace AR,24,8,32.0%,9.55,31.2%,15.4,8.1,Ideal Finisher Seam AR - 32 deliveries faced b...
1,Muzahir Raza,Pace Bowler,24,6,31.2%,9.38,31.0%,14.9,8.4,Lower-Order Hitter - 46 runs off 28 balls to s...
2,Krish Bhagat,Pace AR,24,3,22.4%,10.12,25.0%,18.1,6.5,Operational Reserve - Stabilizing depth covera...
3,Chetan Sakariya,Pace Bowler,24,6,33.5%,9.15,31.5%,15.3,8.2,Elite Youth Target - Left arm wide yorker deat...
4,Aaqib Khan,Pace Bowler,21,2,34.0%,9.22,32.0%,14.8,7.9,High-signal Powerplay-to-Death containment pro...


In [ ]:
print(df["Player Type"].value_counts().to_string())

Player Type
Pace Bowler    23
Spin Bowler    10
Pace AR         2


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

data = df.copy()

# Convert percentage and numeric columns safely
for col in ["Yorker %", "Death Dot Ball %",
            "Death Economy", "Middle Phase SR",
            "Middle Balls / Bndry"]:
    data[col] = pd.to_numeric(
        data[col].astype(str).str.replace("%", "", regex=False),
        errors="coerce"
    )

# Higher percentile = better ability
def ability_score(frame, metrics):
    scores = []
    weights = []

    for col, weight, higher_is_better in metrics:
        values = frame[col]
        percentile = values.rank(pct=True)

        if not higher_is_better:
            percentile = 1 - percentile

        scores.append(percentile * weight)
        weights.append(values.notna() * weight)

    numerator = sum(scores)
    denominator = sum(weights)

    return (numerator / denominator.replace(0, np.nan) * 100).fillna(0)

# 1. Death bowling ability
death_metrics = [
    ("Yorker %", 0.40, True),
    ("Death Economy", 0.35, False),
    ("Death Dot Ball %", 0.25, True)
]

data["Death Score"] = ability_score(data, death_metrics)

# 2. Spin bowling ability - subtype doesn't matter
spin_metrics = [
    ("Middle Phase SR", 0.60, False),
    ("Middle Balls / Bndry", 0.40, True)
]

data["Spin Score"] = ability_score(data, spin_metrics)

# 3. All-phase ability
data["All Phase Score"] = (
    data["Death Score"] * 0.50 +
    data["Spin Score"] * 0.50
)

# Required selection sequence
slots = [
    ("Death Bowler", "Death Score", "Pace"),
    ("Death Bowler", "Death Score", "Pace"),
    ("Spinner", "Spin Score", "Spin"),
    ("Spinner", "Spin Score", "Spin"),
    ("All-Phase Bowler", "All Phase Score", "ALL"),
    ("Death Bowler", "Death Score", "Pace"),
    ("Death Bowler", "Death Score", "Pace"),
    ("Spinner", "Spin Score", "Spin"),
    ("Spinner", "Spin Score", "Spin"),
    ("All-Phase Bowler", "All Phase Score", "ALL")
]

selected = []
used = set()

for rank, (role, score_col, pool_type) in enumerate(slots, 1):

    pool = data[~data["Player Name"].isin(used)].copy()

    if pool_type == "Pace":
        pool = pool[
            pool["Player Type"].str.contains("Pace", case=False, na=False)
        ]
    elif pool_type == "Spin":
        pool = pool[
            pool["Player Type"].str.contains("Spin", case=False, na=False)
        ]

    pool = pool.sort_values(score_col, ascending=False)

    if pool.empty:
        print(f"WARNING: Rank {rank} could not be filled.")
        continue

    player = pool.iloc[0].copy()
    player["Final Rank"] = rank
    player["Selected Role"] = role
    player["Selection Score"] = round(player[score_col], 2)

    selected.append(player)
    used.add(player["Player Name"])

# Final Top 10
final_top10 = pd.DataFrame(selected)

final_top10 = final_top10[
    ["Final Rank", "Player Name", "Player Type", "Age",
     "Matches", "Selected Role", "Selection Score",
     "Death Score", "Spin Score", "All Phase Score",
     "Tactical Roster Utility"]
]

# Export workbook
output = "RR_Bowling_Final_Top_10.xlsx"

with pd.ExcelWriter(output) as writer:
    final_top10.to_excel(writer, sheet_name="Final Top 10", index=False)
    data.sort_values("All Phase Score", ascending=False).to_excel(
        writer, sheet_name="Complete Scorecard", index=False
    )

print("FINAL SELECTION:", len(final_top10), "/ 10")
display(final_top10)

files.download(output)

FINAL SELECTION: 10 / 10


,Final Rank,Player Name,Player Type,Age,Matches,Selected Role,Selection Score,Death Score,Spin Score,All Phase Score,Tactical Roster Utility
3,1,Chetan Sakariya,Pace Bowler,24,6,Death Bowler,95.79,95.785714,64.000000,79.892857,Elite Youth Target - Left arm wide yorker deat...
4,2,Aaqib Khan,Pace Bowler,21,2,Death Bowler,95.00,95.000000,74.571429,84.785714,High-signal Powerplay-to-Death containment pro...
6,3,Manav Suthar,Spin Bowler,24,5,Spinner,98.29,64.214286,98.285714,81.250000,Middle Phase Disruptor - Smashes elite 14.2 ba...
7,4,Musheer Khan,Spin Bowler,21,1,Spinner,95.43,56.214286,95.428571,75.821429,Attacking wrist-spin option beating playoff me...
1,5,Muzahir Raza,Pace Bowler,24,6,All-Phase Bowler,78.18,77.214286,79.142857,78.178571,Lower-Order Hitter - 46 runs off 28 balls to s...
8,6,Rajvardhan Hangargekar,Pace Bowler,24,2,Death Bowler,83.64,83.642857,71.142857,77.392857,High-velocity powerplay-death asset triggering...
14,7,Yudhvir Singh,Pace Bowler,28,9,Death Bowler,77.86,77.857143,54.000000,65.928571,Heavy-ball domestic enforcer option for cross-...
34,8,Murugan Ashwin,Spin Bowler,32,10,Spinner,88.00,44.857143,88.000000,66.428571,Experienced leg-spinner with googly variations
24,9,Himanshu Sharma,Spin Bowler,22,2,Spinner,87.71,45.142857,87.714286,66.428571,Uncapped wrist spinner track asset
5,10,Darshan Nalkande,Pace Bowler,24,6,All-Phase Bowler,75.64,72.142857,79.142857,75.642857,Exceptional under-25 death overs change-of-pac...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

data = df.copy()

# -------------------------------
# 1. CLEAN DATA
# -------------------------------

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    )

numeric_cols = [
    "Age", "Matches", "Yorker %",
    "Death Economy", "Death Dot Ball %",
    "Middle Phase SR", "Middle Balls / Bndry"
]

for col in numeric_cols:
    data[col] = clean_numeric(data[col])

data["Player Type"] = data["Player Type"].astype(str).str.strip()

# -------------------------------
# 2. PERCENTILE SCORING
# -------------------------------

def metric_percentile(series, higher_better=True):
    score = series.rank(pct=True) * 100

    if not higher_better:
        score = 100 - score

    return score

def weighted_score(frame, metrics):
    numerator = pd.Series(0.0, index=frame.index)
    denominator = pd.Series(0.0, index=frame.index)

    for col, weight, higher_better in metrics:
        score = metric_percentile(frame[col], higher_better)

        valid = frame[col].notna()

        numerator += score.fillna(0) * weight
        denominator += valid.astype(float) * weight

    return (numerator / denominator.replace(0, np.nan)).round(2)

# -------------------------------
# 3. DEATH BOWLING SCORE
# -------------------------------

death_pool = data[
    data["Player Type"].str.contains("Pace", case=False, na=False)
].copy()

death_metrics = [
    ("Yorker %", 0.36, True),
    ("Death Economy", 0.315, False),
    ("Death Dot Ball %", 0.225, True),
    ("Matches", 0.10, True)
]

death_pool["Death Score"] = weighted_score(
    death_pool, death_metrics
)

# -------------------------------
# 4. SPIN BOWLING SCORE
# -------------------------------

spin_pool = data[
    data["Player Type"].str.contains("Spin", case=False, na=False)
].copy()

spin_metrics = [
    ("Middle Phase SR", 0.54, False),
    ("Middle Balls / Bndry", 0.36, True),
    ("Matches", 0.10, True)
]

spin_pool["Spin Ability Score"] = weighted_score(
    spin_pool, spin_metrics
)

# -------------------------------
# 5. ALL-PHASE SCORE
# -------------------------------

all_phase_pool = data[
    data["Player Type"].str.contains("Pace AR", case=False, na=False)
].copy()

all_phase_pool["Death Ability"] = weighted_score(
    all_phase_pool, death_metrics
)

all_phase_pool["Middle Control"] = weighted_score(
    all_phase_pool, [
        ("Middle Phase SR", 0.60, False),
        ("Middle Balls / Bndry", 0.40, True)
    ]
)

all_phase_pool["All Phase Score"] = (
    all_phase_pool["Death Ability"] * 0.45 +
    all_phase_pool["Middle Control"] * 0.45 +
    metric_percentile(all_phase_pool["Matches"], True) * 0.10
).round(2)

# -------------------------------
# 6. FINAL TOP 10 SELECTION
# -------------------------------

selection_plan = [
    ("Death Bowler", death_pool, "Death Score"),
    ("Death Bowler", death_pool, "Death Score"),
    ("Spinner", spin_pool, "Spin Ability Score"),
    ("Spinner", spin_pool, "Spin Ability Score"),
    ("All-Phase Bowler", all_phase_pool, "All Phase Score"),
    ("Death Bowler", death_pool, "Death Score"),
    ("Death Bowler", death_pool, "Death Score"),
    ("Spinner", spin_pool, "Spin Ability Score"),
    ("Spinner", spin_pool, "Spin Ability Score"),
    ("All-Phase Bowler", all_phase_pool, "All Phase Score")
]

selected = []
used_players = set()

for rank, (role, pool, score_col) in enumerate(selection_plan, 1):

    candidates = pool[
        (~pool["Player Name"].isin(used_players)) &
        (pool[score_col].notna())
    ].sort_values(score_col, ascending=False)

    if candidates.empty:
        print(f"WARNING: Rank {rank} could not be filled.")
        continue

    player = candidates.iloc[0].copy()

    player["Final Rank"] = rank
    player["Selected Role"] = role
    player["Final Score"] = player[score_col]

    selected.append(player)
    used_players.add(player["Player Name"])

# -------------------------------
# 7. EXPORT FINAL WORKBOOK
# -------------------------------

final_top10 = pd.DataFrame(selected)

if not final_top10.empty:
    final_top10 = final_top10.sort_values("Final Rank")

output = "RR_Bowling_Top10_Revised_Model.xlsx"

with pd.ExcelWriter(output) as writer:
    final_top10.to_excel(
        writer, sheet_name="Final Top 10", index=False
    )

    death_pool.sort_values(
        "Death Score", ascending=False
    ).to_excel(writer, sheet_name="Death Rankings", index=False)

    spin_pool.sort_values(
        "Spin Ability Score", ascending=False
    ).to_excel(writer, sheet_name="Spin Rankings", index=False)

    all_phase_pool.sort_values(
        "All Phase Score", ascending=False
    ).to_excel(writer, sheet_name="All Phase Rankings", index=False)

print("FINAL PLAYERS SELECTED:", len(final_top10), "/ 10")
display(final_top10)

files.download(output)

FINAL PLAYERS SELECTED: 10 / 10


,Player Name,Player Type,Age,Matches,Yorker %,Death Economy,Death Dot Ball %,Middle Phase SR,Middle Balls / Bndry,Tactical Roster Utility,Death Score,Final Rank,Selected Role,Final Score,Spin Ability Score,Death Ability,Middle Control,All Phase Score
3,Chetan Sakariya,Pace Bowler,24,6,33.5,9.15,31.5,15.3,8.2,Elite Youth Target - Left arm wide yorker deat...,94.00,1,Death Bowler,94.00,NaN,NaN,NaN,NaN
4,Aaqib Khan,Pace Bowler,21,2,34.0,9.22,32.0,14.8,7.9,High-signal Powerplay-to-Death containment pro...,89.68,2,Death Bowler,89.68,NaN,NaN,NaN,NaN
6,Manav Suthar,Spin Bowler,24,5,0.0,9.12,31.5,14.0,9.3,Middle Phase Disruptor - Smashes elite 14.2 ba...,NaN,3,Spinner,91.10,91.1,NaN,NaN,NaN
7,Musheer Khan,Spin Bowler,21,1,0.0,9.25,31.0,14.2,9.1,Attacking wrist-spin option beating playoff me...,NaN,4,Spinner,77.60,77.6,NaN,NaN,NaN
0,Raj Bawa,Pace AR,24,8,32.0,9.55,31.2,15.4,8.1,Ideal Finisher Seam AR - 32 deliveries faced b...,NaN,5,All-Phase Bowler,79.41,NaN,84.25,70.0,79.41
14,Yudhvir Singh,Pace Bowler,28,9,31.5,9.35,30.2,15.4,7.8,Heavy-ball domestic enforcer option for cross-...,84.30,6,Death Bowler,84.30,NaN,NaN,NaN,NaN
8,Rajvardhan Hangargekar,Pace Bowler,24,2,32.0,9.30,30.5,15.0,8.1,High-velocity powerplay-death asset triggering...,81.22,7,Death Bowler,81.22,NaN,NaN,NaN,NaN
33,Mayank Dagar,Spin Bowler,30,8,0.0,9.18,31.0,15.0,9.0,Highly restrictive left-arm orthodox option,NaN,8,Spinner,63.80,63.8,NaN,NaN,NaN
34,Murugan Ashwin,Spin Bowler,32,10,0.0,9.32,29.0,14.7,8.6,Experienced leg-spinner with googly variations,NaN,9,Spinner,62.20,62.2,NaN,NaN,NaN
2,Krish Bhagat,Pace AR,24,3,22.4,10.12,25.0,18.1,6.5,Operational Reserve - Stabilizing depth covera...,NaN,10,All-Phase Bowler,29.41,NaN,34.25,20.0,29.41


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [123]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
